In [ ]:
# =========================================================
# 0. INSTALL / IMPORTS / CONFIG
# =========================================================
# Kaggle usually has many packages preinstalled, but Save Version logs are easier to debug
# when all installs stay in the first cell.
!pip -q install -U transformers accelerate sentencepiece bitsandbytes faiss-cpu pyarrow pandas numpy flagembedding tqdm

import os, json, re, math, gc, zipfile, sqlite3, random, shutil, time
from pathlib import Path
from collections import defaultdict, Counter, OrderedDict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = BASE_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Default to the new one-query smoke phase. Use PIPELINE_PHASE=advanced_retrieval
# for the full/test cache run, and keep generate/merge for compatibility.
PIPELINE_PHASE = os.environ.get('PIPELINE_PHASE', 'smoke_query').lower()
RUN_MODE = os.environ.get('RUN_MODE', 'test').lower()  # test | full
TEST_N = int(os.environ.get('TEST_N', '5'))
SAVE_EVERY = int(os.environ.get('SAVE_EVERY', '10'))

EMBED_MODEL = os.environ.get('EMBED_MODEL', 'BAAI/bge-m3')
RERANK_MODEL = os.environ.get('RERANK_MODEL', 'BAAI/bge-reranker-v2-m3')
GEN_MODEL = os.environ.get('GEN_MODEL', 'Qwen/Qwen2.5-7B-Instruct')
PLANNER_MODEL = os.environ.get('PLANNER_MODEL', GEN_MODEL)

# Edit these two values directly for local/Kaggle smoke tests, or override with env vars.
DEFAULT_SMOKE_QUERY_ID = 1128
DEFAULT_SMOKE_QUERY_TEXT = ''
SMOKE_QUERY_ID = int(os.environ.get('SMOKE_QUERY_ID', str(DEFAULT_SMOKE_QUERY_ID)))
SMOKE_QUERY_TEXT = os.environ.get('SMOKE_QUERY_TEXT', DEFAULT_SMOKE_QUERY_TEXT).strip()
SMOKE_QUERY_IDS = [int(x.strip()) for x in os.environ.get('SMOKE_QUERY_IDS', '').split(',') if x.strip()]

CFG = {
    'test_filename': 'R2AIStage1DATA.json',
    'chunk_store_filename': 'chunk_store.sqlite',
    'chunk_meta_slim_filename': 'chunk_meta_slim.parquet',
    'faiss_filename': f"faiss_index__{EMBED_MODEL.replace('/','_')}.index",
    'embed_meta_filename': f"embed_model_meta__{EMBED_MODEL.replace('/','_')}.json",
    'retrieval_cache_base_name': f"retrieval_cache_decomp_anchor_v2__{EMBED_MODEL.replace('/','_')}.jsonl",
    'query_plan_cache_path': str(OUT_DIR / f"query_plan_cache_anchor_v2__{PLANNER_MODEL.replace('/','_')}.jsonl"),
    'smoke_query_debug_path': str(OUT_DIR / 'smoke_query_anchor_v2_debug.json'),
    'hyde_cache_path': str(OUT_DIR / f"hyde_cache__{GEN_MODEL.replace('/','_')}.jsonl"),
    'results_path': str(OUT_DIR / 'results.json'),
    'zip_path': str(OUT_DIR / 'submission.zip'),
    'progress_path': str(OUT_DIR / f"progress_state_decomp_anchor_v2__{EMBED_MODEL.replace('/','_')}__{GEN_MODEL.replace('/','_')}.json"),
    'rrf_k': int(os.environ.get('RRF_K', '60')),
    'lexical_mode': os.environ.get('LEXICAL_MODE', 'off').lower(),  # off | fts_fast | fts_ranked | postings
    'localize_artifacts': os.environ.get('LOCALIZE_ARTIFACTS', '1') == '1',
    'dense_query_batch_size': int(os.environ.get('DENSE_QUERY_BATCH_SIZE', '64')),
    'rerank_batch_size': int(os.environ.get('RERANK_BATCH_SIZE', '48')),
    'bm25_topk': int(os.environ.get('BM25_TOPK', '40')),
    'dense_topk': int(os.environ.get('DENSE_TOPK', '100')),
    'candidate_topk': int(os.environ.get('CANDIDATE_TOPK', '96')),
    'rerank_topk': int(os.environ.get('RERANK_TOPK', '48')),
    'article_context_topk': int(os.environ.get('ARTICLE_CONTEXT_TOPK', '16')),
    'article_context_max_topk': int(os.environ.get('ARTICLE_CONTEXT_MAX_TOPK', '24')),
    'gen_context_topk': int(os.environ.get('GEN_CONTEXT_TOPK', '6')),
    'gen_chunk_char_limit': int(os.environ.get('GEN_CHUNK_CHAR_LIMIT', '900')),
    'gen_max_input_tokens': int(os.environ.get('GEN_MAX_INPUT_TOKENS', '3072')),
    'gen_load_in_4bit': os.environ.get('GEN_LOAD_IN_4BIT', '0') == '1',
    'answer': os.environ.get('ANSWER', '1').strip().lower() not in {'0', 'false', 'no', 'off'},
    'use_llm_planner': os.environ.get('USE_LLM_PLANNER', '1') == '1',
    'planner_load_in_4bit': os.environ.get('PLANNER_LOAD_IN_4BIT', '0') == '1',
    'planner_allow_cpu': os.environ.get('PLANNER_ALLOW_CPU', '0').strip().lower() in {'1', 'true', 'yes', 'on'},
    'planner_max_new_tokens': int(os.environ.get('PLANNER_MAX_NEW_TOKENS', '512')),
    'planner_max_atomic': int(os.environ.get('PLANNER_MAX_ATOMIC', '4')),
    'planner_min_overlap': float(os.environ.get('PLANNER_MIN_OVERLAP', '0.18')),
    'use_must_terms_variant': os.environ.get('USE_MUST_TERMS_VARIANT', '0') == '1',
    'submit_article_max': int(os.environ.get('SUBMIT_ARTICLE_MAX', '12')),
    'submit_article_max_simple': int(os.environ.get('SUBMIT_ARTICLE_MAX_SIMPLE', '2')),
    'submit_article_max_medium': int(os.environ.get('SUBMIT_ARTICLE_MAX_MEDIUM', '4')),
    'submit_article_max_complex': int(os.environ.get('SUBMIT_ARTICLE_MAX_COMPLEX', '12')),
    'candidate_article_debug_topk': int(os.environ.get('CANDIDATE_ARTICLE_DEBUG_TOPK', '24')),
    'diag_topk': int(os.environ.get('DIAG_TOPK', '8')),
    'retrieval_shard_count': int(os.environ.get('RETRIEVAL_SHARD_COUNT', '1')),
    'retrieval_shard_index': int(os.environ.get('RETRIEVAL_SHARD_INDEX', '0')),
    'gen_shard_count': int(os.environ.get('GEN_SHARD_COUNT', '1')),
    'gen_shard_index': int(os.environ.get('GEN_SHARD_INDEX', '0')),
}
if CFG['retrieval_shard_count'] <= 1:
    CFG['retrieval_cache_advanced_path'] = str(OUT_DIR / CFG['retrieval_cache_base_name'])
else:
    CFG['retrieval_cache_advanced_path'] = str(
        OUT_DIR / f"retrieval_cache_decomp_anchor_v2_shard{CFG['retrieval_shard_index']}_of_{CFG['retrieval_shard_count']}__{EMBED_MODEL.replace('/','_')}.jsonl"
    )

PHASES = {'smoke_query', 'advanced_retrieval', 'generate', 'merge', 'all'}
assert PIPELINE_PHASE in PHASES, f'PIPELINE_PHASE must be one of {sorted(PHASES)}'
assert RUN_MODE in {'test', 'full'}, "RUN_MODE must be test or full"
assert 0 <= CFG['retrieval_shard_index'] < CFG['retrieval_shard_count']
assert 0 <= CFG['gen_shard_index'] < CFG['gen_shard_count']

print(json.dumps({
    'phase': PIPELINE_PHASE,
    'run_mode': RUN_MODE,
    'test_n': TEST_N,
    'smoke_query_id': SMOKE_QUERY_ID,
    'smoke_query_ids': SMOKE_QUERY_IDS,
    'smoke_query_text_set': bool(SMOKE_QUERY_TEXT),
    'embed_model': EMBED_MODEL,
    'rerank_model': RERANK_MODEL,
    'gen_model': GEN_MODEL,
    'planner_model': PLANNER_MODEL,
    **CFG,
}, ensure_ascii=False, indent=2))


In [ ]:
# =========================================================
# 0.5 ARTIFACT DISCOVERY + COMMON HELPERS
# =========================================================
def candidate_roots():
    roots = []
    if Path('/kaggle/input').exists():
        roots.append(Path('/kaggle/input'))
    roots.append(OUT_DIR)
    roots.append(BASE_DIR)
    return roots

def find_first(pattern):
    matches = []
    for root in candidate_roots():
        if root.exists():
            matches.extend(root.glob(f'**/{pattern}'))
    matches = sorted({p.resolve() for p in matches if p.is_file()})
    return matches[0] if matches else None

def find_all(pattern):
    matches = []
    for root in candidate_roots():
        if root.exists():
            matches.extend(root.glob(f'**/{pattern}'))
    return sorted({p.resolve() for p in matches if p.is_file()})

TEST_PATH = find_first(CFG['test_filename']) or Path('glrag-testset') / CFG['test_filename']
READ_PATHS = {
    'test_path': str(TEST_PATH),
    'chunk_store_path': str(find_first(CFG['chunk_store_filename']) or OUT_DIR / CFG['chunk_store_filename']),
    'chunk_meta_slim_path': str(find_first(CFG['chunk_meta_slim_filename']) or OUT_DIR / CFG['chunk_meta_slim_filename']),
    'faiss_path': str(find_first(CFG['faiss_filename']) or OUT_DIR / CFG['faiss_filename']),
    'embed_meta_path': str(find_first(CFG['embed_meta_filename']) or OUT_DIR / CFG['embed_meta_filename']),
    'retrieval_cache_advanced_path': str(find_first(Path(CFG['retrieval_cache_advanced_path']).name) or CFG['retrieval_cache_advanced_path']),
    'query_plan_cache_path': str(find_first(Path(CFG['query_plan_cache_path']).name) or CFG['query_plan_cache_path']),
    'hyde_cache_path': str(find_first(Path(CFG['hyde_cache_path']).name) or CFG['hyde_cache_path']),
    'progress_path': str(find_first(Path(CFG['progress_path']).name) or CFG['progress_path']),
}

assert Path(READ_PATHS['test_path']).exists(), f"Missing test set: {READ_PATHS['test_path']}"

_word_re = re.compile(r'\w+', re.UNICODE)
STOPWORDS = {
    'và', 'hoặc', 'của', 'các', 'những', 'một', 'này', 'đó', 'thì', 'là', 'có', 'bị', 'được',
    'phải', 'cho', 'về', 'trong', 'ngoài', 'theo', 'nếu', 'khi', 'như', 'để', 'với', 'từ', 'ra',
    'sao', 'gì', 'nào', 'bao', 'nhiêu', 'trường', 'hợp', 'cần', 'muốn', 'hỏi', 'tôi', 'công', 'ty'
}

LEGAL_EXPANSIONS = [
    (r'\bưu đãi\b', 'hỗ trợ miễn giảm chính sách ưu đãi'),
    (r'\bhỗ trợ\b', 'chính sách hỗ trợ doanh nghiệp nhỏ và vừa'),
    (r'\bđấu thầu\b', 'lựa chọn nhà thầu ưu đãi trong lựa chọn nhà thầu'),
    (r'\bphạt|xử lý|vi phạm\b', 'xử phạt vi phạm hành chính biện pháp khắc phục hậu quả'),
    (r'\bgiữ\b.*\b(bằng|văn bằng|chứng chỉ|giấy tờ)\b', 'giữ bản chính giấy tờ tùy thân văn bằng chứng chỉ của người lao động'),
    (r'\bhợp đồng lao động\b', 'giao kết thực hiện chấm dứt hợp đồng lao động'),
    (r'\bthuế\b', 'thuế thu nhập doanh nghiệp thuế giá trị gia tăng miễn giảm thuế'),
    (r'\bbảo hiểm\b', 'bảo hiểm xã hội bảo hiểm y tế bảo hiểm thất nghiệp'),
    (r'\bhồ sơ\b', 'thành phần hồ sơ tài liệu chứng cứ đơn yêu cầu'),
    (r'\bthủ tục\b', 'trình tự thủ tục hồ sơ cơ quan có thẩm quyền'),
    (r'\bthời hạn\b', 'thời hạn thời hiệu số ngày làm việc'),
    (r'\bquyền tác giả|phần mềm|sao chép\b', 'quyền tác giả chương trình máy tính hành vi xâm phạm thiệt hại chứng cứ'),
]

def normalize_text(text):
    text = str(text).replace('\u200b', ' ').replace('\ufeff', ' ')
    return re.sub(r'\s+', ' ', text).strip()

def normalize_key(text):
    return normalize_text(text).lower()

def tokenize_lexical(text, keep_stopwords=False):
    toks = _word_re.findall(normalize_text(text).lower())
    toks = [t for t in toks if len(t) >= 2]
    if not keep_stopwords:
        toks = [t for t in toks if t not in STOPWORDS]
    return toks

def sqlite_connect(path, readonly=False):
    path = Path(path)
    if readonly:
        return sqlite3.connect(f'file:{path.as_posix()}?mode=ro', uri=True)
    return sqlite3.connect(path)

def write_jsonl(path, record):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

def read_jsonl(path):
    rows = []
    p = Path(path)
    if not p.exists():
        return rows
    with open(p, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def localize_artifact(path, label=None):
    src_path = Path(path)
    if not CFG.get('localize_artifacts', True):
        return str(src_path)
    try:
        src_resolved = src_path.resolve()
    except Exception:
        src_resolved = src_path
    if '/kaggle/input/' not in src_resolved.as_posix():
        return str(src_path)
    local_dir = Path('/tmp/r2ai_localized_artifacts') if Path('/tmp').exists() else BASE_DIR / 'localized_artifacts'
    local_dir.mkdir(parents=True, exist_ok=True)
    dst = local_dir / src_path.name
    if not dst.exists() or dst.stat().st_size != src_path.stat().st_size:
        print(f'Copying artifact to fast local disk: {label or src_path.name}')
        shutil.copy2(src_path, dst)
    return str(dst)

def load_test_df():
    with open(READ_PATHS['test_path'], 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    assert {'id', 'question'}.issubset(df.columns)
    assert df['id'].is_unique
    if RUN_MODE == 'test':
        df = df.head(TEST_N).copy()
    return df

print('READ_PATHS =', json.dumps(READ_PATHS, ensure_ascii=False, indent=2))


In [ ]:
# =========================================================
# 1. ARTIFACT VALIDATION + LOAD SLIM META
# =========================================================
def validate_and_load_artifacts():
    required = ['chunk_store_path', 'chunk_meta_slim_path', 'faiss_path', 'embed_meta_path']
    missing = [k for k in required if not Path(READ_PATHS[k]).exists()]
    assert not missing, json.dumps({'missing': missing, 'paths': READ_PATHS}, ensure_ascii=False, indent=2)

    for key in required:
        READ_PATHS[key] = localize_artifact(READ_PATHS[key], key)

    conn = sqlite_connect(READ_PATHS['chunk_store_path'], readonly=True)
    sqlite_rows = conn.execute('SELECT COUNT(*) FROM chunks').fetchone()[0]
    sqlite_meta = dict(conn.execute('SELECT key, value FROM artifact_meta').fetchall())
    conn.close()

    chunk_meta = pd.read_parquet(READ_PATHS['chunk_meta_slim_path'])
    model_meta = json.loads(Path(READ_PATHS['embed_meta_path']).read_text(encoding='utf-8'))
    assert 'chunk_text' not in chunk_meta.columns
    assert len(chunk_meta) == sqlite_rows, (len(chunk_meta), sqlite_rows)
    assert int(model_meta['count']) == len(chunk_meta), (model_meta.get('count'), len(chunk_meta))
    assert model_meta['embed_model'] == EMBED_MODEL, (model_meta['embed_model'], EMBED_MODEL)
    if 'embed_model' in sqlite_meta:
        assert sqlite_meta['embed_model'] == EMBED_MODEL, (sqlite_meta['embed_model'], EMBED_MODEL)
    if 'row_idx' in chunk_meta.columns:
        expected = np.arange(len(chunk_meta))
        assert np.array_equal(chunk_meta['row_idx'].to_numpy(), expected), 'chunk_meta row_idx is not contiguous; baseline FAISS lookup expects iloc(row_idx)'

    print({'meta_rows': len(chunk_meta), 'sqlite_rows': sqlite_rows, 'sqlite_meta': sqlite_meta, 'model_meta': model_meta})
    return chunk_meta, sqlite_meta, model_meta

if PIPELINE_PHASE in {'smoke_query', 'advanced_retrieval', 'all'}:
    CHUNK_META, SQLITE_META, MODEL_META = validate_and_load_artifacts()
else:
    CHUNK_META, SQLITE_META, MODEL_META = None, None, None


In [ ]:
# =========================================================
# 2. LLM QUERY DECOMPOSITION + COMPLEXITY PLANNING
# =========================================================
QUESTION_TYPES = {
    'procedure', 'condition', 'rights_obligations', 'sanction',
    'support_incentive', 'scenario', 'deadline', 'comparison',
    'definition_listing', 'other'
}
COMPLEXITIES = {'simple', 'medium', 'complex'}

def anchor_plain(text):
    import unicodedata
    text = unicodedata.normalize('NFD', normalize_text(text).lower())
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('\u0111', 'd')

def _append_unique(values, value):
    value = normalize_text(value)
    if value and value not in values:
        values.append(value)

def _plain_has_any(plain, needles):
    return any(needle in plain for needle in needles)


def build_rule_domain_profile(question, must_terms=None, raw_anchor_terms=None):
    must_terms = must_terms or []
    raw_anchor_terms = raw_anchor_terms or []
    haystack = ' '.join([question] + list(must_terms) + list(raw_anchor_terms))
    plain = anchor_plain(haystack)
    labels, anchors, preferred_law_ids = [], [], []
    preferred_title_terms, negative_title_terms, soft_negative_law_ids = [], [], []

    def add_label(label):
        if label not in labels:
            labels.append(label)

    def add_anchor(text):
        _append_unique(anchors, text)

    def add_pref_law(law_id):
        if law_id not in preferred_law_ids:
            preferred_law_ids.append(law_id)

    def add_pref_title(text):
        _append_unique(preferred_title_terms, text)

    def add_negative_title(text):
        _append_unique(negative_title_terms, text)

    def add_soft_negative_law(law_id):
        if law_id not in soft_negative_law_ids:
            soft_negative_law_ids.append(law_id)

    has_copyright = _plain_has_any(plain, [
        'quyen tac gia', 'quyen lien quan', 'phan mem', 'chuong trinh may tinh', 'sao chep',
        'xam pham quyen tac gia'
    ])
    has_industrial = _plain_has_any(plain, ['so huu cong nghiep', 'nhan hieu', 'sang che', 'kieu dang cong nghiep'])
    if has_copyright:
        add_label('copyright_software')
        for term in [
            'quyền tác giả', 'phần mềm', 'sao chép', 'cho thuê',
            'chương trình máy tính', 'xâm phạm quyền tác giả'
        ]:
            if anchor_plain(term) in plain:
                add_anchor(term)
        add_pref_law('50/2005/QH11')
        add_pref_law('17/2023/NĐ-CP')
        for term in ['quyền tác giả', 'quyền liên quan', 'sở hữu trí tuệ']:
            add_pref_title(term)
        if not has_industrial:
            add_negative_title('sở hữu công nghiệp')
            add_negative_title('giống cây trồng')
            add_soft_negative_law('65/2023/NĐ-CP')
            add_soft_negative_law('99/2013/NĐ-CP')
            add_soft_negative_law('11/2015/TT-BKHCN')

    if _plain_has_any(plain, ['hai quan', 'nhap khau', 'kiem soat']):
        add_label('customs_control')
        add_anchor('hải quan')
        add_anchor('kiểm soát')
        add_anchor('nhập khẩu')
        add_pref_law('13/2015/TT-BTC')
        add_pref_law('50/2005/QH11')
        add_pref_title('hải quan')
        add_pref_title('kiểm soát')

    if _plain_has_any(plain, ['giam dinh', 'giam dinh vien']):
        add_label('copyright_assessment')
        add_anchor('giám định')
        add_pref_law('15/2012/TT-BVHTTDL')
        add_pref_law('17/2023/NĐ-CP')
        add_pref_law('105/2006/NĐ-CP')
        add_pref_title('giám định quyền tác giả')
        add_negative_title('chuyển giao công nghệ')
        add_negative_title('thương mại')

    if _plain_has_any(plain, ['nguoi tieu dung', 'de bi ton thuong']):
        add_label('consumer_protection')
        add_anchor('người tiêu dùng')
        add_anchor('dễ bị tổn thương')
        add_pref_law('19/2023/QH15')
        add_pref_title('người tiêu dùng')

    if _plain_has_any(plain, ['doanh nghiep nho va vua', 'dnnvv']) and _plain_has_any(plain, ['dau thau', 'lua chon nha thau']):
        add_label('small_business_procurement')
        add_anchor('doanh nghiệp nhỏ và vừa')
        add_anchor('đấu thầu')
        add_pref_law('04/2017/QH14')
        add_pref_law('22/2023/QH15')
        add_pref_title('hỗ trợ doanh nghiệp nhỏ và vừa')
        add_pref_title('đấu thầu')

    for term in raw_anchor_terms:
        add_anchor(term)

    return {
        'labels': labels,
        'anchor_terms': anchors[:12],
        'preferred_law_ids': preferred_law_ids[:12],
        'preferred_title_terms': preferred_title_terms[:12],
        'negative_title_terms': negative_title_terms[:12],
        'soft_negative_law_ids': soft_negative_law_ids[:12],
    }

def is_generic_atomic_question(text, domain_profile=None):
    plain = anchor_plain(text)
    generic_markers = [
        'tai lieu', 'chung cu', 'ho so', 'don yeu cau', 'hop dong', 'tranh chap', 'xu ly'
    ]
    if not _plain_has_any(plain, generic_markers):
        return False
    domain_profile = domain_profile or {}
    anchors = domain_profile.get('anchor_terms', [])
    return not any(anchor_plain(anchor) in plain for anchor in anchors)

def repair_generic_atomic_question(text, domain_profile):
    text = normalize_text(text)
    if not is_generic_atomic_question(text, domain_profile):
        return text
    anchors = [a for a in domain_profile.get('anchor_terms', []) if normalize_text(a)]
    if not anchors:
        return text
    suffix = ' về ' + ', '.join(anchors[:3])
    if anchor_plain(suffix) in anchor_plain(text):
        return text
    if text.endswith('?'):
        return text[:-1].rstrip() + suffix + '?'
    return text + suffix

def lexical_jaccard(a, b):
    aa = set(tokenize_lexical(a, keep_stopwords=False))
    bb = set(tokenize_lexical(b, keep_stopwords=False))
    if not aa or not bb:
        return 0.0
    return len(aa & bb) / max(len(aa | bb), 1)

def atomic_questions_too_similar(items):
    if len(items) < 2:
        return False
    pairs = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            pairs.append(lexical_jaccard(items[i], items[j]))
    return bool(pairs) and max(pairs) >= 0.62

def dedupe_atomic_questions(items, max_items=None):
    out = []
    for item in items:
        item = normalize_text(item)
        if not item:
            continue
        if any(normalize_key(item) == normalize_key(x) or lexical_jaccard(item, x) >= 0.78 for x in out):
            continue
        out.append(item)
        if max_items and len(out) >= max_items:
            break
    return out

def facet_profile(
    facet_id,
    label,
    query,
    anchor_terms=None,
    preferred_law_ids=None,
    preferred_title_terms=None,
    target_terms=None,
    negative_terms=None,
    priority=1.0,
):
    return {
        'facet_id': str(facet_id),
        'label': normalize_text(label),
        'query': normalize_text(query),
        'anchor_terms': [normalize_text(x) for x in (anchor_terms or []) if normalize_text(x)],
        'preferred_law_ids': [str(x).strip() for x in (preferred_law_ids or []) if str(x).strip()],
        'preferred_title_terms': [normalize_text(x) for x in (preferred_title_terms or []) if normalize_text(x)],
        'target_terms': [normalize_text(x) for x in (target_terms or []) if normalize_text(x)],
        'negative_terms': [normalize_text(x) for x in (negative_terms or []) if normalize_text(x)],
        'priority': float(priority),
    }

def sanitize_facet_profiles(values, max_items=None):
    max_items = max_items or max(6, CFG.get('planner_max_atomic', 4))
    out, seen = [], set()
    if not isinstance(values, list):
        return out
    for raw in values:
        if not isinstance(raw, dict):
            continue
        facet_id = normalize_key(str(raw.get('facet_id') or raw.get('label') or raw.get('query') or 'facet'))
        facet_id = re.sub(r'[^a-z0-9_]+', '_', facet_id).strip('_')[:64] or 'facet'
        query = normalize_text(raw.get('query') or raw.get('text') or raw.get('label') or '')
        label = normalize_text(raw.get('label') or query or facet_id)
        key = facet_id + '|' + normalize_key(query or label)
        if not query or key in seen:
            continue
        out.append(facet_profile(
            facet_id=facet_id,
            label=label,
            query=query,
            anchor_terms=sanitize_list_of_strings(raw.get('anchor_terms', []), max_items=8, max_chars=80),
            preferred_law_ids=sanitize_list_of_strings(raw.get('preferred_law_ids', []), max_items=8, max_chars=40),
            preferred_title_terms=sanitize_list_of_strings(raw.get('preferred_title_terms', []), max_items=8, max_chars=80),
            target_terms=sanitize_list_of_strings(raw.get('target_terms', []), max_items=12, max_chars=80),
            negative_terms=sanitize_list_of_strings(raw.get('negative_terms', []), max_items=8, max_chars=80),
            priority=float(raw.get('priority', 1.0) or 1.0),
        ))
        seen.add(key)
        if len(out) >= max_items:
            break
    return out

def build_rule_facet_profiles(question, domain_profile):
    plain = anchor_plain(question)
    labels = set(domain_profile.get('labels', []))
    facets = []

    def add(facet):
        key = facet['facet_id']
        if key not in {x['facet_id'] for x in facets}:
            facets.append(facet)

    if 'small_business_procurement' in labels:
        add(facet_profile(
            'small_business_support',
            'Ưu đãi hỗ trợ doanh nghiệp nhỏ và vừa',
            'Ưu đãi, hỗ trợ dành cho doanh nghiệp nhỏ và vừa theo pháp luật hỗ trợ doanh nghiệp nhỏ và vừa?',
            anchor_terms=['doanh nghiệp nhỏ và vừa', 'ưu đãi', 'hỗ trợ'],
            preferred_law_ids=['04/2017/QH14'],
            preferred_title_terms=['hỗ trợ doanh nghiệp nhỏ và vừa'],
            target_terms=['doanh nghiệp nhỏ và vừa', 'hỗ trợ', 'ưu đãi', 'chính sách hỗ trợ'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.15,
        ))
        add(facet_profile(
            'procurement_preference',
            'Ưu đãi trong đấu thầu',
            'Ưu đãi đối với doanh nghiệp nhỏ và vừa trong lựa chọn nhà thầu, đấu thầu?',
            anchor_terms=['doanh nghiệp nhỏ và vừa', 'đấu thầu', 'lựa chọn nhà thầu'],
            preferred_law_ids=['22/2023/QH15'],
            preferred_title_terms=['đấu thầu'],
            target_terms=['ưu đãi', 'đấu thầu', 'nhà thầu', 'lựa chọn nhà thầu', 'doanh nghiệp nhỏ và vừa'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.14,
        ))

    if 'copyright_software' in labels:
        multi_domain = bool(labels & {'customs_control', 'copyright_assessment', 'consumer_protection'})
        explicit_software = _plain_has_any(plain, ['phan mem', 'chuong trinh may tinh', 'sao chep', 'cho thue'])
        if (not multi_domain) or explicit_software:
            add(facet_profile(
                'software_property_rights',
                'Quyền tài sản với chương trình máy tính',
                'Quyền tác giả và quyền tài sản đối với chương trình máy tính, phần mềm được quy định thế nào?',
                anchor_terms=['quyền tác giả', 'phần mềm', 'chương trình máy tính'],
                preferred_law_ids=['50/2005/QH11'],
                preferred_title_terms=['sở hữu trí tuệ', 'quyền tác giả'],
                target_terms=['quyền tài sản', 'chương trình máy tính', 'phần mềm', 'sao chép', 'cho thuê'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.12,
            ))
            if _plain_has_any(plain, ['sao chep', 'cho thue', 'xam pham']):
                add(facet_profile(
                    'copyright_infringement',
                    'Hành vi xâm phạm quyền tác giả phần mềm',
                    'Hành vi sao chép, cho thuê trái phép phần mềm xâm phạm quyền tác giả như thế nào?',
                    anchor_terms=['quyền tác giả', 'phần mềm', 'sao chép', 'cho thuê'],
                    preferred_law_ids=['50/2005/QH11'],
                    preferred_title_terms=['sở hữu trí tuệ', 'quyền tác giả'],
                    target_terms=['xâm phạm', 'sao chép', 'cho thuê', 'quyền tác giả', 'chương trình máy tính'],
                    negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                    priority=1.11,
                ))
        if _plain_has_any(plain, ['ton that', 'mat khach hang', 'co hoi kinh doanh', 'thiet hai', 'boi thuong']):
            add(facet_profile(
                'damage_business_opportunity',
                'Thiệt hại và cơ hội kinh doanh',
                'Cách xác định thiệt hại và tổn thất cơ hội kinh doanh do xâm phạm quyền tác giả phần mềm?',
                anchor_terms=['quyền tác giả', 'phần mềm', 'thiệt hại', 'cơ hội kinh doanh'],
                preferred_law_ids=['50/2005/QH11', '17/2023/NĐ-CP'],
                preferred_title_terms=['sở hữu trí tuệ', 'quyền tác giả'],
                target_terms=['thiệt hại', 'tổn thất', 'cơ hội kinh doanh', 'bồi thường', 'mất khách hàng'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.10,
            ))
        if _plain_has_any(plain, ['tu bao ve', 'yeu cau xu ly', 'bien phap bao ve', 'xu ly xam pham']):
            add(facet_profile(
                'self_protection_request',
                'Quyền tự bảo vệ và yêu cầu xử lý',
                'Quyền tự bảo vệ, yêu cầu xử lý xâm phạm và biện pháp bảo vệ quyền sở hữu trí tuệ được quy định thế nào?',
                anchor_terms=['quyền tác giả', 'xâm phạm', 'yêu cầu xử lý'],
                preferred_law_ids=['50/2005/QH11'],
                preferred_title_terms=['sở hữu trí tuệ'],
                target_terms=['quyền tự bảo vệ', 'yêu cầu xử lý', 'biện pháp bảo vệ', 'xâm phạm quyền sở hữu trí tuệ'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.08,
            ))
        if _plain_has_any(plain, ['tai lieu', 'chung cu', 'don yeu cau', 'xu ly']):
            add(facet_profile(
                'evidence_request_docs',
                'Tài liệu chứng cứ yêu cầu xử lý',
                'Tài liệu, chứng cứ cần chuẩn bị khi yêu cầu xử lý hành vi xâm phạm quyền tác giả đối với phần mềm?',
                anchor_terms=['quyền tác giả', 'phần mềm', 'tài liệu', 'chứng cứ'],
                preferred_law_ids=['17/2023/NĐ-CP', '50/2005/QH11'],
                preferred_title_terms=['quyền tác giả', 'sở hữu trí tuệ'],
                target_terms=['tài liệu', 'chứng cứ', 'đơn yêu cầu', 'yêu cầu xử lý', 'xâm phạm quyền tác giả'],
                negative_terms=['sở hữu công nghiệp', 'nhãn hiệu', 'sáng chế'],
                priority=1.06,
            ))

    if 'customs_control' in labels:
        add(facet_profile(
            'customs_financial_guarantee',
            'Kiểm soát hải quan và bảo đảm tài chính',
            'Nghĩa vụ bảo đảm tài chính khi yêu cầu hải quan kiểm soát hàng hóa nghi xâm phạm quyền tác giả?',
            anchor_terms=['hải quan', 'kiểm soát', 'nhập khẩu', 'bảo đảm tài chính'],
            preferred_law_ids=['13/2015/TT-BTC', '50/2005/QH11'],
            preferred_title_terms=['hải quan', 'sở hữu trí tuệ'],
            target_terms=['hải quan', 'kiểm soát', 'hàng hóa', 'bảo đảm tài chính', 'tạm dừng làm thủ tục'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.16,
        ))
    if 'copyright_assessment' in labels:
        add(facet_profile(
            'assessment_contract',
            'Hợp đồng giám định quyền tác giả',
            'Hợp đồng giám định quyền tác giả, quyền liên quan cần có những nội dung chính nào?',
            anchor_terms=['giám định', 'hợp đồng giám định', 'quyền tác giả'],
            preferred_law_ids=['15/2012/TT-BVHTTDL', '17/2023/NĐ-CP', '105/2006/NĐ-CP'],
            preferred_title_terms=['giám định quyền tác giả', 'quyền tác giả'],
            target_terms=['giám định', 'hợp đồng giám định', 'giám định viên', 'nội dung hợp đồng'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.15,
        ))
    if 'consumer_protection' in labels:
        add(facet_profile(
            'consumer_dispute',
            'Tranh chấp với người tiêu dùng dễ bị tổn thương',
            'Trách nhiệm giải quyết tranh chấp khi đối tượng bị xâm phạm là người tiêu dùng dễ bị tổn thương?',
            anchor_terms=['người tiêu dùng', 'dễ bị tổn thương', 'tranh chấp'],
            preferred_law_ids=['19/2023/QH15'],
            preferred_title_terms=['người tiêu dùng'],
            target_terms=['người tiêu dùng', 'dễ bị tổn thương', 'tranh chấp', 'trách nhiệm'],
            negative_terms=['chuyển giao công nghệ', 'thương mại'],
            priority=1.14,
        ))

    limit = max(6, CFG.get('planner_max_atomic', 4))
    return sanitize_facet_profiles(facets, max_items=limit)

def build_rule_legal_facets(question, domain_profile):
    return dedupe_atomic_questions(
        [f.get('query', '') for f in build_rule_facet_profiles(question, domain_profile)],
        max_items=max(6, CFG.get('planner_max_atomic', 4)),
    )

def merge_facet_profiles(rule_facets, raw_facets):
    out, seen = [], set()
    for facet in list(rule_facets or []) + list(raw_facets or []):
        if not isinstance(facet, dict):
            continue
        key = facet.get('facet_id', '') + '|' + normalize_key(facet.get('query', ''))
        if not key.strip('|') or key in seen:
            continue
        out.append(facet)
        seen.add(key)
    return out[:max(6, CFG.get('planner_max_atomic', 4))]

def enrich_query_plan(question, plan):
    plan = dict(plan or {})
    must_terms = sanitize_list_of_strings(plan.get('must_have_terms', []), max_items=12, max_chars=80)
    raw_anchors = sanitize_list_of_strings(plan.get('anchor_terms', []), max_items=12, max_chars=80)
    raw_facets = sanitize_facet_profiles(plan.get('facet_profiles', []), max_items=max(6, CFG.get('planner_max_atomic', 4)))
    domain_profile = build_rule_domain_profile(question, must_terms=must_terms, raw_anchor_terms=raw_anchors)
    complexity = plan.get('complexity', 'medium')
    rule_facet_profiles = build_rule_facet_profiles(question, domain_profile)
    facet_profiles = merge_facet_profiles(rule_facet_profiles, raw_facets)
    rule_facets = dedupe_atomic_questions(
        [f.get('query', '') for f in facet_profiles] + sanitize_list_of_strings(plan.get('legal_facets', []), max_items=6, max_chars=220),
        max_items=max(6, CFG.get('planner_max_atomic', 4)),
    )

    atomic = sanitize_list_of_strings(plan.get('atomic_questions', []), max_items=CFG['planner_max_atomic'], max_chars=240)
    if complexity == 'simple':
        atomic = []
    else:
        atomic = [repair_generic_atomic_question(item, domain_profile) for item in atomic]
        labels = set(domain_profile.get('labels', []))
        force_rule_first = (
            complexity == 'complex'
            or 'small_business_procurement' in labels
            or atomic_questions_too_similar(atomic)
            or len(rule_facet_profiles) >= 2
        )
        if rule_facets and force_rule_first:
            atomic = dedupe_atomic_questions(rule_facets + atomic, max_items=CFG['planner_max_atomic'])
        elif rule_facets and len(atomic) < min(2, CFG['planner_max_atomic']):
            atomic = dedupe_atomic_questions(atomic + rule_facets, max_items=CFG['planner_max_atomic'])
        else:
            atomic = dedupe_atomic_questions(atomic, max_items=CFG['planner_max_atomic'])
        if not atomic and complexity in {'medium', 'complex'}:
            atomic = dedupe_atomic_questions(rule_facets or [normalize_text(question)], max_items=CFG['planner_max_atomic'])

    plan['atomic_questions'] = atomic[:CFG['planner_max_atomic']]
    plan['must_have_terms'] = must_terms or fallback_must_terms_inline(question)
    plan['anchor_terms'] = domain_profile.get('anchor_terms', [])
    plan['legal_facets'] = rule_facets
    plan['facet_profiles'] = facet_profiles
    plan['domain_profile'] = domain_profile
    plan['generic_atomic_questions'] = [a for a in atomic if is_generic_atomic_question(a, domain_profile)]
    return plan

def fallback_must_terms_inline(question, max_terms=10):
    toks = tokenize_lexical(question, keep_stopwords=False)
    seen = []
    for tok in toks:
        if tok not in seen:
            seen.append(tok)
    return seen[:max_terms]


SLIDE_REFERENCE_NOTES = {
    1577: {
        'label': 'Dễ',
        'expected': ['16/2012/QH13|Luật Quảng cáo|Điều 17'],
        'note': '1 văn bản, 1 điều; không cần phân rã, không multi-hop.',
    },
    2: {
        'label': 'Trung bình',
        'expected': ['04/2017/QH14|Luật Hỗ trợ DNNVV|Điều 13', '22/2023/QH15|Luật Đấu thầu|Điều 10'],
        'note': 'Cross-doc 2 văn bản: ưu đãi DNNVV + ưu đãi trong đấu thầu.',
    },
    1128: {
        'label': 'Khó',
        'expected': ['Luật SHTT Điều 20, 22, 28, 198, 204, 205', 'NĐ 17/2023 Điều 66, 73, 75, 76'],
        'note': '3 yêu cầu: hành vi xâm phạm, tổn thất/cơ hội kinh doanh, tài liệu/chứng cứ xử lý.',
    },
    1720: {
        'label': 'Tình huống thực tế',
        'expected': ['SHTT', 'kiểm soát hải quan', 'giám định', 'bảo vệ người tiêu dùng'],
        'note': '4 vế đan xen nhiều lĩnh vực; 3 văn bản, 4 điều.',
    },
}

SMOKE_EXPECTED_ARTICLES = {
    1577: [('16/2012/QH13', 'Điều 17')],
    2: [('04/2017/QH14', 'Điều 13'), ('22/2023/QH15', 'Điều 10')],
    1128: [
        ('50/2005/QH11', 'Điều 20'), ('50/2005/QH11', 'Điều 22'), ('50/2005/QH11', 'Điều 28'),
        ('50/2005/QH11', 'Điều 198'), ('50/2005/QH11', 'Điều 204'), ('50/2005/QH11', 'Điều 205'),
        ('17/2023/NĐ-CP', 'Điều 66'), ('17/2023/NĐ-CP', 'Điều 73'),
        ('17/2023/NĐ-CP', 'Điều 75'), ('17/2023/NĐ-CP', 'Điều 76'),
    ],
}

SMOKE_EXPECTED_DOMAINS = {
    1720: {
        'shtt': ['50/2005/QH11', '17/2023/NĐ-CP'],
        'hai_quan': ['13/2015/TT-BTC', 'hải quan', 'kiểm soát'],
        'giam_dinh': ['15/2012/TT-BVHTTDL', 'giám định'],
        'nguoi_tieu_dung': ['19/2023/QH15', 'người tiêu dùng'],
    },
}

def split_question_clauses_rule(question):
    q = normalize_text(question)
    pieces = re.split(
        r'[;?。]+|\s+(?:đồng thời|ngoài ra|bên cạnh đó|trong trường hợp|nếu|khi|và nếu|và phải|và cần|đặc biệt nếu)\s+',
        q,
        flags=re.IGNORECASE,
    )
    out = []
    for piece in pieces:
        piece = normalize_text(piece.strip(' ,.-:'))
        if 25 <= len(piece) <= 300:
            out.append(piece)
    if len(out) <= 1 and len(q) > 120:
        for piece in re.split(r',\s+| và ', q):
            piece = normalize_text(piece.strip(' ,.-:'))
            if 25 <= len(piece) <= 260:
                out.append(piece)
    seen, deduped = set(), []
    for piece in out:
        key = normalize_key(piece)
        if key not in seen and key != normalize_key(q):
            seen.add(key)
            deduped.append(piece)
    return deduped[:CFG['planner_max_atomic']]

def detect_question_type(question):
    q = normalize_key(question)
    if re.search(r'\b(quy định|liệt kê|bao gồm|gồm|những|các)\b.*\b(nào|gì)\b', q) and not re.search(r'\b(thủ tục|hồ sơ|điều kiện|xử phạt|vi phạm|trách nhiệm|nghĩa vụ|ưu đãi|hỗ trợ)\b', q):
        return 'definition_listing'
    if re.search(r'\b(thủ tục|hồ sơ|tài liệu|chứng cứ|đơn yêu cầu|chuẩn bị)\b', q):
        return 'procedure'
    if re.search(r'\b(điều kiện|yêu cầu|tiêu chí|đáp ứng)\b', q):
        return 'condition'
    if re.search(r'\b(quyền|nghĩa vụ|trách nhiệm)\b', q):
        return 'rights_obligations'
    if re.search(r'\b(phạt|xử phạt|vi phạm|xử lý|khắc phục)\b', q):
        return 'sanction'
    if re.search(r'\b(hỗ trợ|ưu đãi|miễn|giảm)\b', q):
        return 'support_incentive'
    if re.search(r'\b(thời hạn|bao lâu|khi nào|mấy ngày)\b', q):
        return 'deadline'
    if re.search(r'\b(khác gì|khác biệt|so sánh)\b', q):
        return 'comparison'
    if len(tokenize_lexical(question, keep_stopwords=True)) >= 45:
        return 'scenario'
    return 'other'

def is_simple_listing_query(question):
    q = normalize_key(question)
    toks = tokenize_lexical(question, keep_stopwords=True)
    if len(toks) > 16:
        return False
    if re.search(r'\b(thủ tục|hồ sơ|điều kiện|xử phạt|vi phạm|khắc phục|trách nhiệm|nghĩa vụ|ưu đãi|hỗ trợ|đấu thầu|bồi thường)\b', q):
        return False
    listing_patterns = [
        r'\bquy định\s+(?:những|các)?\s*.+\s+nào\b',
        r'\b(?:những|các)\s+.+\s+nào\b',
        r'\b.+\s+bao gồm\s+(?:những|các)?\s+gì\b',
    ]
    return any(re.search(pat, q) for pat in listing_patterns)

def refine_query_plan_heuristics(question, plan):
    plan = dict(plan or {})
    if is_simple_listing_query(question):
        plan['complexity'] = 'simple'
        plan['atomic_questions'] = []
        plan['question_type'] = 'definition_listing'
        note = 'force_simple_listing'
        existing = normalize_text(plan.get('rationale_short', ''))
        plan['rationale_short'] = (existing + '; ' + note).strip('; ')[:260]
    return plan

def fallback_complexity(question, clauses=None):
    toks = tokenize_lexical(question, keep_stopwords=True)
    q = normalize_key(question)
    clauses = clauses if clauses is not None else split_question_clauses_rule(question)
    multi_markers = len(re.findall(r'\b(và|đồng thời|ngoài ra|nếu|khi|hồ sơ|chứng cứ|xử lý|khắc phục|nghĩa vụ|trách nhiệm|thiệt hại|giám định)\b', q))
    domain_markers = 0
    for pat in [
        r'doanh nghiệp nhỏ và vừa|dnnvv',
        r'đấu thầu|lựa chọn nhà thầu',
        r'quyền tác giả|sở hữu trí tuệ|phần mềm|sao chép',
        r'hải quan|nhập khẩu|xuất khẩu',
        r'giám định',
        r'người tiêu dùng|dễ bị tổn thương',
        r'thuế|đất đai|mặt bằng',
        r'lao động|hợp đồng lao động|bằng cấp|chứng chỉ',
    ]:
        if re.search(pat, q, flags=re.IGNORECASE):
            domain_markers += 1
    if len(toks) >= 50 or len(clauses) >= 3 or multi_markers >= 5:
        return 'complex'
    if domain_markers >= 3:
        return 'complex'
    if len(toks) >= 20 or len(clauses) >= 2 or multi_markers >= 2 or domain_markers >= 2:
        return 'medium'
    return 'simple'

def fallback_must_terms(question, max_terms=10):
    toks = tokenize_lexical(question, keep_stopwords=False)
    seen = []
    for tok in toks:
        if tok not in seen:
            seen.append(tok)
    return seen[:max_terms]

def fallback_query_plan(question, reason='rule_fallback'):
    clauses = split_question_clauses_rule(question)
    complexity = fallback_complexity(question, clauses)
    atomic = clauses if complexity != 'simple' else []
    if not atomic and complexity in {'medium', 'complex'}:
        atomic = [normalize_text(question)]
    plan = {
        'complexity': complexity,
        'atomic_questions': atomic[:CFG['planner_max_atomic']],
        'must_have_terms': fallback_must_terms(question),
        'question_type': detect_question_type(question),
        'rationale_short': reason,
        'planner_fallback': True,
        'planner_error': reason,
        'raw_plan_text': '',
    }
    return enrich_query_plan(question, refine_query_plan_heuristics(question, plan))

def extract_json_object(text):
    text = str(text).strip()
    start = text.find('{')
    end = text.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('No JSON object found in planner output')
    return json.loads(text[start:end + 1])

def token_overlap_ratio(candidate, original):
    cand = set(tokenize_lexical(candidate, keep_stopwords=False))
    orig = set(tokenize_lexical(original, keep_stopwords=False))
    if not cand or not orig:
        return 0.0
    return len(cand & orig) / max(len(cand), 1)

def has_forbidden_new_citation(text, original):
    original_has_citation = bool(re.search(r'\b(Điều\s+\d+|Luật\s+\d+|Nghị định\s+\d+|Thông tư\s+\d+|Nghị quyết\s+\d+|\d+/\d{4}/[A-ZĐ-]+)\b', original, flags=re.IGNORECASE))
    if original_has_citation:
        return False
    return bool(re.search(r'\b(Điều\s+\d+|\d+/\d{4}/[A-ZĐ-]+)\b', str(text), flags=re.IGNORECASE))

def sanitize_list_of_strings(values, max_items=8, max_chars=220):
    out, seen = [], set()
    if not isinstance(values, list):
        return out
    for value in values:
        text = normalize_text(str(value))[:max_chars].strip(' -')
        key = normalize_key(text)
        if text and key not in seen:
            out.append(text)
            seen.add(key)
        if len(out) >= max_items:
            break
    return out

def validate_query_plan(raw_plan, question, raw_text=''):
    fallback = fallback_query_plan(question)
    if not isinstance(raw_plan, dict):
        fallback['raw_plan_text'] = raw_text
        fallback['planner_error'] = 'planner returned non-dict'
        return fallback

    complexity = str(raw_plan.get('complexity', '')).strip().lower()
    if complexity not in COMPLEXITIES:
        complexity = fallback['complexity']

    qtype = str(raw_plan.get('question_type', '')).strip().lower()
    if qtype not in QUESTION_TYPES:
        qtype = fallback['question_type']

    atomic = sanitize_list_of_strings(raw_plan.get('atomic_questions', []), max_items=CFG['planner_max_atomic'])
    clean_atomic = []
    for item in atomic:
        if normalize_key(item) == normalize_key(question):
            continue
        if has_forbidden_new_citation(item, question):
            continue
        if token_overlap_ratio(item, question) < CFG['planner_min_overlap']:
            continue
        clean_atomic.append(item)

    if not clean_atomic and complexity in {'medium', 'complex'}:
        clean_atomic = fallback['atomic_questions']
    clean_atomic = clean_atomic[:CFG['planner_max_atomic']]

    terms = sanitize_list_of_strings(raw_plan.get('must_have_terms', []), max_items=12, max_chars=60)
    terms = [t for t in terms if not has_forbidden_new_citation(t, question)]
    if not terms:
        terms = fallback['must_have_terms']

    plan = {
        'complexity': complexity,
        'atomic_questions': clean_atomic,
        'must_have_terms': terms,
        'anchor_terms': sanitize_list_of_strings(raw_plan.get('anchor_terms', []), max_items=12, max_chars=80),
        'legal_facets': sanitize_list_of_strings(raw_plan.get('legal_facets', []), max_items=max(6, CFG['planner_max_atomic']), max_chars=220),
        'facet_profiles': sanitize_facet_profiles(raw_plan.get('facet_profiles', []), max_items=max(6, CFG['planner_max_atomic'])),
        'domain_profile': raw_plan.get('domain_profile', {}) if isinstance(raw_plan.get('domain_profile', {}), dict) else {},
        'question_type': qtype,
        'rationale_short': normalize_text(raw_plan.get('rationale_short', ''))[:260],
        'planner_fallback': False,
        'planner_error': '',
        'raw_plan_text': raw_text,
    }
    return enrich_query_plan(question, refine_query_plan_heuristics(question, plan))

class LLMQueryPlanner:
    def __init__(self, model_name=PLANNER_MODEL):
        self.enabled = bool(CFG.get('use_llm_planner', True))
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.disabled_reason = ''
        if not self.enabled:
            self.disabled_reason = 'USE_LLM_PLANNER=0'
            print({'planner': 'disabled', 'fallback': True})
            return
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        if device != 'cuda' and not CFG.get('planner_allow_cpu', False):
            self.enabled = False
            self.disabled_reason = 'planner_cpu_fallback'
            print({
                'planner': 'cpu_fallback',
                'fallback': True,
                'reason': 'CUDA is not available; refusing to load planner LLM on CPU. Set PLANNER_ALLOW_CPU=1 to override.',
                'planner_model': model_name,
            })
            return
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        load_kwargs = {'trust_remote_code': True}
        if device == 'cuda' and CFG.get('planner_load_in_4bit', True):
            from transformers import BitsAndBytesConfig
            load_kwargs.update({
                'device_map': 'auto',
                'torch_dtype': torch.float16,
                'quantization_config': BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_type='nf4',
                    bnb_4bit_use_double_quant=True,
                ),
            })
        elif device == 'cuda':
            load_kwargs.update({'torch_dtype': torch.float16, 'device_map': 'auto'})
        else:
            load_kwargs.update({'torch_dtype': torch.float32})
        print({'planner_model': model_name, 'planner_device': device, 'planner_4bit': bool(device == 'cuda' and CFG.get('planner_load_in_4bit', True))})
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        self.model.eval()

    def build_prompt(self, question):
        system = (
            'Bạn là bộ phân tích truy vấn cho hệ thống truy hồi văn bản pháp luật Việt Nam. '
            'Chỉ phân rã ý hỏi, không trả lời câu hỏi, không suy đoán số điều, không bịa tên văn bản. '
            'Luôn trả về đúng một JSON object hợp lệ.'
        )
        user = f'''
Câu hỏi:
{question}

Hãy trả về JSON theo schema:
{{
  "complexity": "simple|medium|complex",
  "atomic_questions": ["mệnh đề truy hồi độc lập, tối đa 4"],
  "must_have_terms": ["thuật ngữ bắt buộc lấy từ hoặc bám rất sát câu hỏi"],
  "anchor_terms": ["domain anchors that every atomic question must preserve"],
  "legal_facets": ["separate legal retrieval facets, not paraphrases"],
  "facet_profiles": [{{"facet_id": "stable_id", "label": "legal facet", "anchor_terms": [], "preferred_law_ids": [], "preferred_title_terms": [], "target_terms": [], "negative_terms": [], "priority": 1.0}}],
  "domain_profile": {{"labels": ["domain labels if clear"]}},
  "question_type": "procedure|condition|rights_obligations|sanction|support_incentive|scenario|deadline|comparison|definition_listing|other",
  "rationale_short": "lý do ngắn, không quá 1 câu"
}}

Quy tắc:
- Every atomic question must preserve specific domain anchors from the original question, such as copyright, software, customs, assessment, vulnerable consumers, small and medium enterprises, or procurement.
- Do not shorten an atomic question into a generic form like documents/evidence, contract contents, dispute handling, or request processing when the original question contains a specific domain.
- legal_facets should split legal aspects, not paraphrase the same question multiple times.
- facet_profiles should name the legal facets that need separate slot coverage; keep law/title preferences only when directly implied by the question.
- simple: 1 vấn đề, thường 1 văn bản/1 điều.
- medium: 2 ý hoặc cross-doc nhẹ.
- complex: nhiều mệnh đề, multi-hop, nhiều lĩnh vực/văn bản.
- atomic_questions phải giữ đúng ý gốc, không thêm vấn đề mới.
- Không nêu "Điều X", mã luật, tên văn bản cụ thể nếu câu hỏi không nêu.
- Không dùng markdown, không giải thích ngoài JSON.
'''
        if hasattr(self.tokenizer, 'apply_chat_template'):
            return self.tokenizer.apply_chat_template([
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': user},
            ], tokenize=False, add_generation_prompt=True)
        return system + '\n\n' + user + '\n\nJSON:'

    def plan(self, question):
        question = normalize_text(question)
        if not self.enabled:
            return fallback_query_plan(question, reason=self.disabled_reason or 'planner_disabled')
        try:
            prompt = self.build_prompt(question)
            inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(self.model.device)
            with torch.no_grad():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=CFG['planner_max_new_tokens'],
                    temperature=0.0,
                    do_sample=False,
                    repetition_penalty=1.01,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
            gen_ids = out[0][inputs['input_ids'].shape[1]:]
            raw_text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
            raw_plan = extract_json_object(raw_text)
            return validate_query_plan(raw_plan, question, raw_text=raw_text)
        except Exception as e:
            plan = fallback_query_plan(question, reason='planner_exception')
            plan['planner_error'] = repr(e)
            return plan

def add_variant(variants, text, kind, weight, dense_only=False, lexical_only=False):
    text = normalize_text(text)
    if not text:
        return
    key = normalize_key(text)
    if key in {v['key'] for v in variants}:
        return
    variants.append({
        'text': text,
        'kind': kind,
        'weight': float(weight),
        'dense_only': bool(dense_only),
        'lexical_only': bool(lexical_only),
        'key': key,
    })

def must_terms_query(query_plan):
    terms = [normalize_text(t) for t in query_plan.get('must_have_terms', []) if normalize_text(t)]
    return ' '.join(terms[:16])

def build_query_variants(question, query_plan=None):
    query_plan = query_plan or fallback_query_plan(question)
    complexity = query_plan.get('complexity', 'medium')
    variants = []
    add_variant(variants, question, 'original', 1.00)

    if complexity == 'simple':
        atomic_weights = [0.86]
        facet_weight = 0.00
        must_weight = 0.58
    elif complexity == 'complex':
        atomic_weights = [0.96, 0.93, 0.90, 0.87]
        facet_weight = 0.88
        must_weight = 0.68
    else:
        atomic_weights = [0.94, 0.90, 0.86]
        facet_weight = 0.91
        must_weight = 0.64

    for idx, atomic in enumerate(query_plan.get('atomic_questions', [])[:CFG['planner_max_atomic']]):
        weight = atomic_weights[min(idx, len(atomic_weights) - 1)]
        add_variant(variants, atomic, f'atomic_{idx + 1}', weight)

    if complexity != 'simple':
        for idx, facet in enumerate(query_plan.get('facet_profiles', [])[:max(6, CFG.get('planner_max_atomic', 4))]):
            text = facet.get('query') or facet.get('label') or ''
            facet_id = re.sub(r'[^a-z0-9_]+', '_', normalize_key(facet.get('facet_id', f'facet_{idx + 1}'))).strip('_') or f'facet_{idx + 1}'
            priority = float(facet.get('priority', 1.0) or 1.0)
            weight = min(0.97, facet_weight + 0.02 * max(priority - 1.0, 0.0))
            add_variant(variants, text, f'facet_{idx + 1}_{facet_id}', weight)

    if CFG.get('use_must_terms_variant', True):
        term_query = must_terms_query(query_plan)
        if term_query and normalize_key(term_query) != normalize_key(question):
            add_variant(variants, term_query, 'must_terms', must_weight, lexical_only=True)

    return variants

def load_or_build_query_plan_cache(test_df):
    out_path = Path(CFG['query_plan_cache_path'])
    read_path = find_first(out_path.name)
    if not out_path.exists() and read_path and Path(read_path) != out_path:
        out_path.write_text(Path(read_path).read_text(encoding='utf-8'), encoding='utf-8')

    cache = {}
    for rec in read_jsonl(out_path):
        try:
            cache[int(rec['id'])] = rec['query_plan']
        except Exception:
            pass

    missing = [(int(r['id']), str(r['question'])) for _, r in test_df.iterrows() if int(r['id']) not in cache]
    if not missing:
        return cache

    print({'phase': 'query_planning', 'missing': len(missing), 'use_llm_planner': CFG['use_llm_planner']})
    planner = LLMQueryPlanner()
    pbar = tqdm(total=len(missing), desc='Query decomposition', unit='query', dynamic_ncols=True)
    for qid, question in missing:
        plan = planner.plan(question)
        rec = {'id': qid, 'question': question, 'query_plan': plan}
        write_jsonl(out_path, rec)
        cache[qid] = plan
        pbar.update(1)
    pbar.close()
    del planner
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return cache


In [ ]:
# =========================================================
# 3. DECOMP RETRIEVAL: LLM QUERY PLAN + RRF + ARTICLE SELECTION
# =========================================================
def fetch_contexts_from_store(row_indices, include_text=True):
    if not row_indices:
        return []
    row_indices = [int(x) for x in row_indices]
    placeholders = ','.join(['?'] * len(row_indices))
    cols = 'row_idx, chunk_id, doc_uid, law_id, ten_van_ban, dieu_so, chunk_text' if include_text else 'row_idx, chunk_id, doc_uid, law_id, ten_van_ban, dieu_so'
    conn = sqlite_connect(READ_PATHS['chunk_store_path'], readonly=True)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(f'SELECT {cols} FROM chunks WHERE row_idx IN ({placeholders})', row_indices).fetchall()
    conn.close()
    by_idx = {int(r['row_idx']): dict(r) for r in rows}
    return [by_idx[i] for i in row_indices if i in by_idx]

def fts_query_from_text(query, max_terms=10):
    toks = tokenize_lexical(query, keep_stopwords=False)
    if not toks:
        toks = tokenize_lexical(query, keep_stopwords=True)
    seen = []
    for tok in toks:
        if tok not in seen:
            seen.append(tok)
    terms = seen[:max_terms]
    if not terms:
        return ''
    return ' OR '.join('"' + t.replace('"', '""') + '"' for t in terms)

class SQLiteLexicalRetriever:
    def __init__(self, db_path, mode=None):
        self.db_path = db_path
        self.mode = (mode or CFG.get('lexical_mode', 'fts_fast')).lower()
        conn = sqlite_connect(db_path, readonly=True)
        self.backend = conn.execute("SELECT value FROM artifact_meta WHERE key='lexical_backend'").fetchone()[0]
        self.n_docs = int(conn.execute("SELECT value FROM artifact_meta WHERE key='rows'").fetchone()[0])
        avg = conn.execute("SELECT value FROM artifact_meta WHERE key='avgdl'").fetchone()
        self.avgdl = float(avg[0]) if avg else 0.0
        conn.close()
        print({'lexical_backend': self.backend, 'lexical_mode': self.mode})

    def search(self, query, topk=40):
        if self.mode in {'off', 'none', '0', 'false'}:
            return []
        if self.backend == 'fts5' and self.mode == 'fts_ranked':
            return self._search_fts_ranked(query, topk)
        if self.backend == 'fts5':
            return self._search_fts_fast(query, topk)
        return self._search_postings(query, topk)

    def _search_fts_fast(self, query, topk):
        q = fts_query_from_text(query)
        if not q:
            return []
        conn = sqlite_connect(self.db_path, readonly=True)
        conn.row_factory = sqlite3.Row
        try:
            rows = conn.execute('''
                SELECT c.row_idx, c.chunk_id, c.doc_uid, c.law_id, c.ten_van_ban, c.dieu_so
                FROM chunks_fts
                JOIN chunks c ON c.row_idx = chunks_fts.rowid
                WHERE chunks_fts MATCH ?
                LIMIT ?
            ''', (q, int(topk))).fetchall()
        except sqlite3.OperationalError as e:
            print('FTS fast query failed:', q, e)
            rows = []
        finally:
            conn.close()
        out = []
        for rank, r in enumerate(rows, start=1):
            item = dict(r)
            item['lexical_score'] = 1.0 / rank
            item['lexical_rank'] = rank
            out.append(item)
        return out

    def _search_fts_ranked(self, query, topk):
        q = fts_query_from_text(query)
        if not q:
            return []
        conn = sqlite_connect(self.db_path, readonly=True)
        conn.row_factory = sqlite3.Row
        try:
            rows = conn.execute('''
                SELECT c.row_idx, c.chunk_id, c.doc_uid, c.law_id, c.ten_van_ban, c.dieu_so, bm25(chunks_fts) AS rank
                FROM chunks_fts
                JOIN chunks c ON c.row_idx = chunks_fts.rowid
                WHERE chunks_fts MATCH ?
                ORDER BY rank
                LIMIT ?
            ''', (q, int(topk))).fetchall()
        except sqlite3.OperationalError as e:
            print('FTS ranked query failed:', q, e)
            rows = []
        finally:
            conn.close()
        out = []
        for rank, r in enumerate(rows, start=1):
            item = dict(r)
            item['lexical_score'] = float(-item.pop('rank'))
            item['lexical_rank'] = rank
            out.append(item)
        return out

    def _search_postings(self, query, topk):
        terms = list(dict.fromkeys(tokenize_lexical(query, keep_stopwords=False)))[:10]
        if not terms:
            return []
        conn = sqlite_connect(self.db_path, readonly=True)
        scores = defaultdict(float)
        k1, b = 1.5, 0.75
        for term in terms:
            stat = conn.execute('SELECT idf FROM term_stats WHERE term=?', (term,)).fetchone()
            if not stat:
                continue
            idf = float(stat[0])
            for row_idx, freq, dl in conn.execute('''
                SELECT p.row_idx, p.freq, d.dl
                FROM term_postings p JOIN doc_lens d ON d.row_idx = p.row_idx
                WHERE p.term=?
            ''', (term,)):
                denom = freq + k1 * (1 - b + b * dl / max(self.avgdl, 1e-9))
                scores[int(row_idx)] += idf * (freq * (k1 + 1)) / denom
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk]
        contexts = fetch_contexts_from_store([idx for idx, _ in top], include_text=False)
        conn.close()
        by_idx = {int(c['row_idx']): c for c in contexts}
        out = []
        for rank, (idx, score) in enumerate(top, start=1):
            item = dict(by_idx.get(idx, {'row_idx': idx}))
            item['lexical_score'] = float(score)
            item['lexical_rank'] = rank
            out.append(item)
        return out

class DenseRetriever:
    def __init__(self, index, chunk_meta, model):
        self.index = index
        self.chunk_meta = chunk_meta.reset_index(drop=True)
        self.model = model

    def encode_queries(self, queries):
        out = self.model.encode(
            [normalize_text(q) for q in queries],
            batch_size=min(max(len(queries), 1), CFG['dense_query_batch_size']),
            max_length=512,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        qemb = out['dense_vecs'].astype('float32')
        qemb = qemb / np.maximum(np.linalg.norm(qemb, axis=1, keepdims=True), 1e-12)
        return np.ascontiguousarray(qemb, dtype='float32')

    def search_queries(self, queries, topk=100):
        if not queries:
            return []
        qemb = self.encode_queries(queries)
        scores, idx = self.index.search(qemb, topk)
        all_results = []
        for qpos in range(len(queries)):
            results = []
            for rank, (i, s) in enumerate(zip(idx[qpos].tolist(), scores[qpos].tolist()), start=1):
                if i < 0:
                    continue
                row = self.chunk_meta.iloc[int(i)]
                results.append({
                    'row_idx': int(i),
                    'chunk_id': str(row.get('chunk_id', '')),
                    'dense_score': float(s),
                    'dense_rank': rank,
                    'doc_uid': str(row.get('doc_uid', '')),
                    'law_id': str(row.get('law_id', '')),
                    'ten_van_ban': str(row.get('ten_van_ban', '')),
                    'dieu_so': str(row.get('dieu_so', '')),
                })
            all_results.append(results)
        return all_results

class Reranker:
    def __init__(self, model_name=RERANK_MODEL):
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.batch_size = CFG['rerank_batch_size']
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True).to(self.device)
        self.model.eval()
        print({'reranker_device': self.device, 'rerank_batch_size': self.batch_size})

    def rerank(self, query, candidates, topk=48):
        if not candidates:
            return []
        pairs = [(query, c.get('chunk_text', '')) for c in candidates]
        scores = []
        with torch.no_grad():
            for i in range(0, len(pairs), self.batch_size):
                batch = pairs[i:i+self.batch_size]
                q, t = zip(*batch)
                inputs = self.tokenizer(list(q), list(t), padding=True, truncation=True, max_length=512, return_tensors='pt').to(self.device)
                out = self.model(**inputs)
                logits = out.logits.squeeze(-1)
                scores.extend(logits.detach().cpu().numpy().tolist())
        reranked = []
        for c, s in zip(candidates, scores):
            item = dict(c)
            item['rerank_score'] = float(s)
            reranked.append(item)
        reranked.sort(key=lambda x: x['rerank_score'], reverse=True)
        return reranked[:topk]

def weighted_rrf_fuse(weighted_rankings, k=60, topk=96):
    scores = defaultdict(float)
    payload = {}
    sources = defaultdict(list)
    for ranking, weight, label in weighted_rankings:
        for rank, item in enumerate(ranking, start=1):
            row_idx = int(item['row_idx'])
            scores[row_idx] += float(weight) / (k + rank)
            payload.setdefault(row_idx, item)
            sources[row_idx].append({'source': label, 'rank': rank, 'weight': float(weight)})
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk]
    out = []
    for row_idx, score in fused:
        item = dict(payload[row_idx])
        item['rrf_score'] = float(score)
        item['source_hits'] = sources[row_idx][:12]
        out.append(item)
    return out

def article_key(c):
    law_id = str(c.get('law_id', '')).strip()
    ten = str(c.get('ten_van_ban', '')).strip()
    dieu = str(c.get('dieu_so', '')).strip()
    if law_id and ten and dieu:
        return f'{law_id}|{ten}|{dieu}'
    return ''

def source_kind(source_label):
    label = str(source_label)
    if ':' in label:
        return label.split(':', 1)[1]
    return label

def canonical_article_key(c):
    law_id = str(c.get('law_id', '')).strip()
    dieu = str(c.get('dieu_so', '')).strip()
    if law_id and dieu:
        return f'{law_id}|{dieu}'
    return article_key(c)


def score_candidate_domain(candidate, query_plan=None):
    query_plan = query_plan or {}
    profile = query_plan.get('domain_profile', {}) or {}
    law_id = str(candidate.get('law_id', '')).strip()
    title = str(candidate.get('ten_van_ban', ''))
    hay = anchor_plain(' '.join([law_id, title, str(candidate.get('dieu_so', ''))]))
    score = 0.0
    reasons = []

    preferred_law_ids = set(profile.get('preferred_law_ids', []))
    soft_negative_law_ids = set(profile.get('soft_negative_law_ids', []))
    if law_id in preferred_law_ids:
        score += 0.75
        reasons.append('preferred_law_id')
    if law_id in soft_negative_law_ids:
        score -= 0.70
        reasons.append('soft_negative_law_id')

    for term in profile.get('preferred_title_terms', []):
        if anchor_plain(term) in hay:
            score += 0.25
            reasons.append('preferred_title:' + term)
            break

    for term in profile.get('negative_title_terms', []):
        if anchor_plain(term) in hay and law_id not in preferred_law_ids:
            score -= 0.80
            reasons.append('negative_title:' + term)
            break

    labels = set(profile.get('labels', []))
    if 'copyright_assessment' in labels and ('chuyen giao cong nghe' in hay or 'thuong mai' in hay) and law_id not in preferred_law_ids:
        score -= 0.55
        reasons.append('assessment_domain_drift')
    if 'copyright_software' in labels and 'so huu cong nghiep' in hay and law_id not in preferred_law_ids:
        score -= 0.80
        reasons.append('industrial_property_drift')
    if not reasons:
        reasons.append('neutral')
    return float(score), reasons[:6]

def is_generic_variant_text(text, query_plan=None):
    profile = (query_plan or {}).get('domain_profile', {})
    try:
        return is_generic_atomic_question(text, profile)
    except Exception:
        plain = anchor_plain(text)
        return _plain_has_any(plain, ['tai lieu', 'chung cu', 'ho so', 'don yeu cau', 'hop dong', 'tranh chap'])

def candidate_facet_haystack(candidate):
    pieces = [
        str(candidate.get('law_id', '')),
        str(candidate.get('ten_van_ban', '')),
        str(candidate.get('dieu_so', '')),
        str(candidate.get('article_key', '')),
        str(candidate.get('chunk_text', ''))[:2400],
    ]
    return anchor_plain(' '.join(pieces))

def score_one_facet(candidate, facet, support_variants=None):
    support_variants = support_variants or []
    law_id = str(candidate.get('law_id', '')).strip()
    hay = candidate_facet_haystack(candidate)
    facet_id = str(facet.get('facet_id', '')).strip()
    rank_score = 0.0
    evidence_score = 0.0
    reasons = []
    preferred_law_ids = set(str(x).strip() for x in facet.get('preferred_law_ids', []) if str(x).strip())
    if law_id and law_id in preferred_law_ids:
        rank_score += 1.10
        reasons.append('preferred_law_id:' + law_id)
    for term in facet.get('preferred_title_terms', []):
        if anchor_plain(term) in hay:
            rank_score += 0.20
            reasons.append('preferred_title:' + term)
            break
    anchor_hits = []
    for term in facet.get('anchor_terms', []):
        if anchor_plain(term) in hay:
            anchor_hits.append(term)
    if anchor_hits:
        rank_score += min(0.30, 0.08 * len(set(anchor_plain(x) for x in anchor_hits)))
        evidence_score += min(0.22, 0.06 * len(set(anchor_plain(x) for x in anchor_hits)))
        reasons.append('anchor_terms:' + ','.join(anchor_hits[:4]))
    target_hits = []
    for term in facet.get('target_terms', []):
        if anchor_plain(term) in hay:
            target_hits.append(term)
    if target_hits:
        rank_score += min(0.90, 0.18 * len(set(anchor_plain(x) for x in target_hits)))
        evidence_score += min(1.25, 0.30 * len(set(anchor_plain(x) for x in target_hits)))
        reasons.append('target_terms:' + ','.join(target_hits[:4]))
    if facet_id and any(facet_id in str(v) for v in support_variants):
        rank_score += 0.45
        evidence_score += 0.55
        reasons.append('facet_variant_support')
    for term in facet.get('negative_terms', []):
        if anchor_plain(term) in hay and law_id not in preferred_law_ids:
            rank_score -= 0.45
            evidence_score -= 0.25
            reasons.append('negative_term:' + term)
            break
    return float(rank_score), float(evidence_score), reasons[:6]

def score_candidate_facets(candidate, query_plan=None, support_variants=None):
    query_plan = query_plan or {}
    support_variants = support_variants or []
    facets = query_plan.get('facet_profiles', []) or []
    if not facets:
        return 0.0, [], [], {}, {}
    matched = []
    reasons = []
    score_map = {}
    evidence_map = {}
    best = 0.0
    for facet in facets:
        facet_id = str(facet.get('facet_id', '')).strip()
        score, evidence_score, facet_reasons = score_one_facet(candidate, facet, support_variants=support_variants)
        score_map[facet_id] = float(score)
        evidence_map[facet_id] = float(evidence_score)
        best = max(best, float(score))
        threshold = 0.45 if facet.get('preferred_law_ids') else 0.55
        if evidence_score >= threshold:
            matched.append(facet_id)
            reasons.append({'facet_id': facet_id, 'score': float(score), 'evidence_score': float(evidence_score), 'reasons': facet_reasons})
    return float(best), matched, reasons[:8], score_map, evidence_map

def aggregate_article_candidates_from_variants(variant_results, query_plan=None):
    groups = OrderedDict()
    for vr in variant_results:
        variant = vr['variant']
        kind = str(variant.get('kind', ''))
        weight = float(variant.get('weight', 1.0))
        for rank, c in enumerate(vr.get('reranked', []), start=1):
            full_key = article_key(c)
            key = canonical_article_key(c)
            if not key:
                continue
            if key not in groups:
                groups[key] = {
                    'canonical_article_key': key,
                    'article_key': full_key,
                    'law_id': str(c.get('law_id', '')).strip(),
                    'ten_van_ban': str(c.get('ten_van_ban', '')).strip(),
                    'dieu_so': str(c.get('dieu_so', '')).strip(),
                    'best_context': dict(c),
                    'best_rerank_score': float(c.get('rerank_score', -1e9)),
                    'best_rrf_score': float(c.get('rrf_score', 0.0)),
                    'best_variant_kind': kind,
                    'best_variant_rank': rank,
                    'support_count': 0,
                    'support_rrf_sum': 0.0,
                    'support_variants': set(),
                    'source_labels': [],
                    'per_variant': {},
                    'generic_atomic_support': False,
                }
            g = groups[key]
            score = float(c.get('rerank_score', -1e9))
            g['support_count'] += 1
            g['support_rrf_sum'] += float(c.get('rrf_score', 0.0))
            g['support_variants'].add(kind)
            g['source_labels'].extend(s.get('source', '') for s in c.get('source_hits', []))
            if str(kind).startswith('atomic_') and is_generic_variant_text(variant.get('text', ''), query_plan):
                g['generic_atomic_support'] = True
            prev = g['per_variant'].get(kind)
            if prev is None or score > prev['best_rerank_score']:
                g['per_variant'][kind] = {
                    'best_rank': rank,
                    'best_rerank_score': score,
                    'best_rrf_score': float(c.get('rrf_score', 0.0)),
                    'weight': weight,
                }
            if score > g['best_rerank_score']:
                g['best_context'] = dict(c)
                g['best_rerank_score'] = score
                g['best_rrf_score'] = float(c.get('rrf_score', 0.0))
                g['best_variant_kind'] = kind
                g['best_variant_rank'] = rank
                g['article_key'] = full_key
                g['ten_van_ban'] = str(c.get('ten_van_ban', '')).strip()

    candidates = []
    for g in groups.values():
        variants = sorted(v for v in g['support_variants'] if v)
        original_support = 'original' in variants
        atomic_coverage_count = len([v for v in variants if v.startswith('atomic_')])
        facet_support_count = len([v for v in variants if v.startswith('facet_')])
        early_bonus = 0.20 / max(int(g['best_variant_rank']), 1)
        original_bonus = 0.24 if original_support else 0.0
        atomic_bonus = 0.18 * min(atomic_coverage_count, 4)
        facet_bonus = 0.12 * min(facet_support_count, 4)
        support_bonus = 0.05 * math.log1p(g['support_count'])
        rrf_bonus = 0.22 * g['support_rrf_sum']
        weak_penalty = -0.20 if variants == ['must_terms'] else 0.0
        domain_score, domain_reasons = score_candidate_domain(g['best_context'], query_plan)
        facet_score, matched_facets, facet_reasons, facet_scores, facet_evidence_scores = score_candidate_facets(g['best_context'], query_plan, support_variants=variants)
        complexity = (query_plan or {}).get('complexity', 'medium')
        domain_weight = 0.0 if complexity == 'simple' else (0.08 if complexity == 'medium' else 0.18)
        facet_weight = 0.0 if complexity == 'simple' else (0.34 if complexity == 'medium' else 0.40)
        generic_penalty = -0.35 if g.get('generic_atomic_support') and not original_support and domain_score < 0.30 and facet_score < 0.90 else 0.0
        article_score_raw = float(g['best_rerank_score'] + early_bonus + original_bonus + atomic_bonus + facet_bonus + support_bonus + rrf_bonus + weak_penalty)
        article_score = float(article_score_raw + domain_weight * domain_score + facet_weight * facet_score + generic_penalty)
        c = dict(g['best_context'])
        c['canonical_article_key'] = g['canonical_article_key']
        c['article_key'] = g['article_key']
        c['article_score'] = article_score
        c['article_score_raw'] = article_score_raw
        c['domain_score'] = float(domain_score)
        c['domain_reasons'] = domain_reasons
        c['facet_score'] = float(facet_score)
        c['best_facet_score'] = float(facet_score)
        c['facet_scores'] = facet_scores
        c['facet_evidence_score'] = float(max(facet_evidence_scores.values()) if facet_evidence_scores else 0.0)
        c['facet_evidence_scores'] = facet_evidence_scores
        c['matched_facets'] = matched_facets
        c['facet_reasons'] = facet_reasons
        c['generic_atomic_support'] = bool(g.get('generic_atomic_support', False))
        c['support_count'] = g['support_count']
        c['support_rrf_sum'] = float(g['support_rrf_sum'])
        c['support_variants'] = variants
        c['source_kinds'] = variants
        c['source_labels'] = sorted(set(g['source_labels']))[:16]
        c['original_support'] = bool(original_support)
        c['atomic_support_count'] = int(atomic_coverage_count)
        c['atomic_coverage_count'] = int(atomic_coverage_count)
        c['facet_support_count'] = int(facet_support_count)
        c['best_variant_kind'] = g['best_variant_kind']
        c['best_variant_rank'] = int(g['best_variant_rank'])
        c['best_variant_rerank_score'] = float(g['best_rerank_score'])
        c['per_variant'] = g['per_variant']
        candidates.append(c)
    candidates.sort(key=lambda x: x['article_score'], reverse=True)
    return candidates

def article_bounds_for_complexity(complexity):
    if complexity == 'simple':
        return {'min_k': 1, 'target_k': 1, 'max_k': min(CFG['submit_article_max_simple'], CFG['submit_article_max'])}
    if complexity == 'complex':
        return {'min_k': 4, 'target_k': 10, 'max_k': min(CFG['submit_article_max_complex'], CFG['submit_article_max'])}
    return {'min_k': 2, 'target_k': 4, 'max_k': min(CFG['submit_article_max_medium'], CFG['submit_article_max'])}

def article_law_family(cand):
    return str(cand.get('law_id', '')).strip()

def add_selected_article(selected, seen, cand, reason, family_counts=None, max_per_family=None):
    key = cand.get('canonical_article_key') or canonical_article_key(cand)
    if not key or key in seen:
        return False
    family = article_law_family(cand)
    count_before = family_counts.get(family, 0) if family_counts is not None else 0
    if family_counts is not None and max_per_family is not None and count_before >= max_per_family:
        return False
    item = dict(cand)
    item.setdefault('selection_reasons', [])
    item['selection_reasons'] = list(item.get('selection_reasons', [])) + [reason]
    item['law_family_count_before_select'] = int(count_before)
    selected.append(item)
    seen.add(key)
    if family_counts is not None:
        family_counts[family] = count_before + 1
    return True

def candidates_for_variant(article_candidates, kind):
    out = [c for c in article_candidates if kind in c.get('per_variant', {})]
    out.sort(key=lambda c: (
        c.get('per_variant', {}).get(kind, {}).get('best_rank', 10**9),
        -float(c.get('best_facet_score', 0.0)),
        -float(c.get('domain_score', 0.0)),
        -float(c.get('per_variant', {}).get(kind, {}).get('best_rerank_score', -1e9)),
    ))
    return out

def facet_profiles_for_selection(query_plan):
    facets = list((query_plan or {}).get('facet_profiles', []) or [])
    facets.sort(key=lambda f: -float(f.get('priority', 1.0) or 1.0))
    return facets

def candidate_allowed_for_selection(cand, complexity, selected_len, min_k, require_facet=False):
    domain_score = float(cand.get('domain_score', 0.0))
    facet_score = float(cand.get('best_facet_score', cand.get('facet_score', 0.0)))
    if complexity == 'simple':
        return True
    if require_facet and facet_score < 0.85:
        return False
    if domain_score <= -0.75 and not cand.get('original_support', False) and facet_score < 1.10 and selected_len >= min_k:
        return False
    if complexity == 'medium':
        return domain_score > -1.00 or cand.get('original_support', False) or facet_score >= 0.85 or selected_len < min_k
    return True

def facet_score_for_candidate(cand, facet):
    facet_id = str(facet.get('facet_id', '')).strip()
    return float((cand.get('facet_scores') or {}).get(facet_id, 0.0))

def facet_evidence_score_for_candidate(cand, facet):
    facet_id = str(facet.get('facet_id', '')).strip()
    return float((cand.get('facet_evidence_scores') or {}).get(facet_id, 0.0))

def candidate_law_preferred_for_facet(cand, facet):
    return str(cand.get('law_id', '')).strip() in set(str(x).strip() for x in facet.get('preferred_law_ids', []) if str(x).strip())

def candidate_matches_facet(cand, facet):
    evidence = facet_evidence_score_for_candidate(cand, facet)
    threshold = 0.45 if facet.get('preferred_law_ids') else 0.55
    return evidence >= threshold

def candidates_for_facet(article_candidates, facet):
    facet_id = str(facet.get('facet_id', '')).strip()
    out = []
    for cand in article_candidates:
        evidence = facet_evidence_score_for_candidate(cand, facet)
        if candidate_matches_facet(cand, facet) or (candidate_law_preferred_for_facet(cand, facet) and evidence >= 0.30):
            out.append(cand)
    out.sort(key=lambda c: (
        -float(candidate_law_preferred_for_facet(c, facet)),
        -facet_evidence_score_for_candidate(c, facet),
        -facet_score_for_candidate(c, facet),
        int(c.get('best_variant_rank', 10**9)),
        -float(c.get('article_score', 0.0)),
    ))
    return out

def preferred_doc_rescue_candidates(article_candidates, query_plan):
    profile = query_plan.get('domain_profile', {}) if query_plan else {}
    preferred = set(profile.get('preferred_law_ids', []))
    if not preferred:
        return []
    out = [c for c in article_candidates if str(c.get('law_id', '')).strip() in preferred]
    out.sort(key=lambda c: (
        -float(c.get('best_facet_score', 0.0)),
        -float(c.get('domain_score', 0.0)),
        -float(c.get('article_score', 0.0)),
        int(c.get('best_variant_rank', 10**9)),
    ))
    return out

def preferred_facet_candidate_exists(article_candidates, facet):
    preferred = set(str(x).strip() for x in facet.get('preferred_law_ids', []) if str(x).strip())
    if not preferred:
        return False
    return any(candidate_law_preferred_for_facet(cand, facet) and candidate_matches_facet(cand, facet) for cand in article_candidates)

def selected_covers_facet(selected, facet, article_candidates=None):
    preferred = set(str(x).strip() for x in facet.get('preferred_law_ids', []) if str(x).strip())
    require_preferred = bool(preferred) and preferred_facet_candidate_exists(article_candidates or [], facet)
    for cand in selected:
        if require_preferred and not candidate_law_preferred_for_facet(cand, facet):
            continue
        if candidate_matches_facet(cand, facet):
            return True
    return False

def selected_covered_facets(selected, query_plan=None, article_candidates=None):
    if query_plan:
        covered = set()
        for facet in facet_profiles_for_selection(query_plan):
            if selected_covers_facet(selected, facet, article_candidates=article_candidates):
                covered.add(str(facet.get('facet_id', '')))
        return covered
    covered = set()
    for cand in selected:
        for facet_id in cand.get('matched_facets', []):
            covered.add(facet_id)
    return covered

def low_value_duplicate_facet(cand, covered_facets):
    matched = set(cand.get('matched_facets', []))
    if not matched:
        return False
    if not matched.issubset(covered_facets):
        return False
    return float(cand.get('best_facet_score', 0.0)) < 1.20 and not cand.get('original_support', False)

def select_facet_coverage(article_candidates, query_plan, complexity, selected, seen, family_counts, max_k, min_k, coverage_debug):
    facets = facet_profiles_for_selection(query_plan)
    for facet in facets:
        if len(selected) >= max_k:
            coverage_debug.append({'facet_id': facet.get('facet_id', ''), 'status': 'skipped', 'reason': 'max_k'})
            break
        facet_id = facet.get('facet_id', '')
        if selected_covers_facet(selected, facet, article_candidates=article_candidates):
            coverage_debug.append({'facet_id': facet_id, 'status': 'covered_before', 'reason': 'already_covered'})
            continue
        added = False
        skip_reasons = []
        facet_candidates = candidates_for_facet(article_candidates, facet)
        for allow_over_cap in ([False, True] if complexity == 'complex' else [True]):
            for cand in facet_candidates:
                key = cand.get('canonical_article_key') or canonical_article_key(cand)
                if key in seen:
                    skip_reasons.append('seen')
                    continue
                if not candidate_matches_facet(cand, facet):
                    skip_reasons.append('low_facet_evidence')
                    continue
                if not candidate_allowed_for_selection(cand, complexity, len(selected), min_k, require_facet=True):
                    skip_reasons.append('domain_or_low_facet')
                    continue
                max_per_family = 2 if (complexity == 'complex' and not allow_over_cap) else None
                if add_selected_article(selected, seen, cand, f'facet_coverage:{facet_id}', family_counts=family_counts, max_per_family=max_per_family):
                    coverage_debug.append({
                        'facet_id': facet_id,
                        'status': 'selected',
                        'article_key': cand.get('article_key', ''),
                        'law_id': cand.get('law_id', ''),
                        'dieu_so': cand.get('dieu_so', ''),
                        'facet_score': facet_score_for_candidate(cand, facet),
                        'facet_evidence_score': facet_evidence_score_for_candidate(cand, facet),
                        'over_family_cap': bool(allow_over_cap),
                    })
                    added = True
                    break
                skip_reasons.append('family_cap')
            if added:
                break
        if not added:
            coverage_debug.append({
                'facet_id': facet_id,
                'status': 'missed',
                'candidate_count': len(facet_candidates),
                'skip_reasons': sorted(set(skip_reasons))[:6],
            })

def select_article_contexts(article_candidates, query_plan, variants=None):
    complexity = query_plan.get('complexity', 'medium')
    bounds = article_bounds_for_complexity(complexity)
    variants = variants or []
    if not article_candidates:
        return [], {'selected_k': 0, 'reason': 'no_candidates', **bounds, 'complexity': complexity, 'coverage_debug': []}

    max_k = min(bounds['max_k'], len(article_candidates))
    min_k = min(bounds['min_k'], max_k)
    selected, seen = [], set()
    family_counts = {}
    coverage_debug = []
    reason = 'facet_coverage_aware_v2'

    if complexity == 'simple':
        add_selected_article(selected, seen, article_candidates[0], 'simple_top1')
        if len(article_candidates) > 1 and len(selected) < max_k:
            top = float(article_candidates[0].get('article_score', 0.0))
            second = float(article_candidates[1].get('article_score', 0.0))
            same_law = article_candidates[0].get('law_id') == article_candidates[1].get('law_id')
            if same_law and second >= top - 0.55:
                add_selected_article(selected, seen, article_candidates[1], 'simple_close_same_law')
    else:
        select_facet_coverage(article_candidates, query_plan, complexity, selected, seen, family_counts, max_k, min_k, coverage_debug)

        original_quota = 1 if complexity == 'medium' else 2
        for cand in candidates_for_variant(article_candidates, 'original')[:original_quota]:
            if len(selected) >= max_k:
                break
            if candidate_allowed_for_selection(cand, complexity, len(selected), min_k):
                add_selected_article(selected, seen, cand, 'original_quota', family_counts=family_counts)

        if complexity == 'complex':
            rescue_added = 0
            for cand in preferred_doc_rescue_candidates(article_candidates, query_plan):
                if len(selected) >= max_k or rescue_added >= 4:
                    break
                if not candidate_allowed_for_selection(cand, complexity, len(selected), min_k):
                    continue
                if low_value_duplicate_facet(cand, selected_covered_facets(selected, query_plan, article_candidates)) and len(selected) >= min_k:
                    continue
                if add_selected_article(selected, seen, cand, 'preferred_doc_rescue', family_counts=family_counts, max_per_family=2):
                    rescue_added += 1

        base_variant_quota = 1 if complexity == 'medium' else 2
        for v in variants:
            kind = str(v.get('kind', ''))
            if not (kind.startswith('atomic_') or kind.startswith('facet_')):
                continue
            added_for_variant = 0
            for cand in candidates_for_variant(article_candidates, kind):
                if len(selected) >= max_k:
                    break
                if not candidate_allowed_for_selection(cand, complexity, len(selected), min_k):
                    continue
                if low_value_duplicate_facet(cand, selected_covered_facets(selected, query_plan, article_candidates)) and len(selected) >= min_k:
                    continue
                if added_for_variant >= base_variant_quota:
                    if not (float(cand.get('best_facet_score', 0.0)) >= 1.10 or cand.get('original_support', False)):
                        continue
                max_per_family = 3 if complexity == 'complex' else None
                if add_selected_article(selected, seen, cand, f'{kind}_quota', family_counts=family_counts, max_per_family=max_per_family):
                    added_for_variant += 1
                if added_for_variant >= base_variant_quota + 1:
                    break

        for cand in article_candidates:
            if len(selected) >= max_k:
                break
            if float(cand.get('domain_score', 0.0)) <= -0.75 and len(selected) >= min_k:
                continue
            if low_value_duplicate_facet(cand, selected_covered_facets(selected, query_plan, article_candidates)) and len(selected) >= min_k:
                continue
            if candidate_allowed_for_selection(cand, complexity, len(selected), min_k):
                add_selected_article(selected, seen, cand, 'score_fill', family_counts=family_counts, max_per_family=3 if complexity == 'complex' else None)

        if len(selected) < min_k:
            for cand in article_candidates:
                if len(selected) >= min_k or len(selected) >= max_k:
                    break
                add_selected_article(selected, seen, cand, 'min_k_backfill', family_counts=family_counts)

    selected.sort(key=lambda x: x.get('article_score', 0.0), reverse=True)
    debug_rows = []
    for rank, c in enumerate(article_candidates[:max(max_k, CFG['candidate_article_debug_topk'])], start=1):
        debug_rows.append({
            'rank': rank,
            'article_key': c.get('article_key', ''),
            'canonical_article_key': c.get('canonical_article_key', ''),
            'article_score': float(c.get('article_score', 0.0)),
            'article_score_raw': float(c.get('article_score_raw', c.get('article_score', 0.0))),
            'domain_score': float(c.get('domain_score', 0.0)),
            'domain_reasons': list(c.get('domain_reasons', [])),
            'facet_score': float(c.get('facet_score', 0.0)),
            'best_facet_score': float(c.get('best_facet_score', 0.0)),
            'facet_evidence_score': float(c.get('facet_evidence_score', 0.0)),
            'facet_evidence_scores': dict(c.get('facet_evidence_scores', {})),
            'matched_facets': list(c.get('matched_facets', [])),
            'facet_reasons': list(c.get('facet_reasons', [])),
            'generic_atomic_support': bool(c.get('generic_atomic_support', False)),
            'rerank_score': float(c.get('rerank_score', 0.0)),
            'best_variant_kind': c.get('best_variant_kind', ''),
            'best_variant_rank': int(c.get('best_variant_rank', 0)),
            'original_support': bool(c.get('original_support', False)),
            'atomic_coverage_count': int(c.get('atomic_coverage_count', 0)),
            'facet_support_count': int(c.get('facet_support_count', 0)),
            'support_variants': list(c.get('support_variants', [])),
            'law_family_count_before_select': c.get('law_family_count_before_select', ''),
        })
    return selected, {
        'complexity': complexity,
        'selected_k': int(len(selected)),
        'reason': reason,
        **bounds,
        'covered_facets': sorted(selected_covered_facets(selected, query_plan, article_candidates)),
        'coverage_debug': coverage_debug,
        'score_debug': debug_rows,
    }

def build_gen_contexts(article_contexts):
    gen_contexts = []
    for c in article_contexts:
        item = dict(c)
        item['chunk_text'] = str(item.get('chunk_text', ''))[:CFG['gen_chunk_char_limit']]
        gen_contexts.append(item)
        if len(gen_contexts) >= CFG['gen_context_topk']:
            break
    return gen_contexts

def make_relevant_lists(article_contexts):
    docs, articles = [], []
    seen_docs, seen_articles = set(), set()
    for c in article_contexts:
        law_id = str(c.get('law_id', '')).strip()
        ten = str(c.get('ten_van_ban', '')).strip()
        dieu = str(c.get('dieu_so', '')).strip()
        if law_id and ten:
            d = f'{law_id}|{ten}'
            if d not in seen_docs:
                seen_docs.add(d)
                docs.append(d)
        if law_id and ten and dieu:
            a = f'{law_id}|{ten}|{dieu}'
            if a not in seen_articles:
                seen_articles.add(a)
                articles.append(a)
    return docs, articles

def compact_hit(hit):
    return {
        'row_idx': int(hit.get('row_idx', -1)),
        'law_id': str(hit.get('law_id', '')),
        'ten_van_ban': str(hit.get('ten_van_ban', '')),
        'dieu_so': str(hit.get('dieu_so', '')),
        'dense_score': float(hit.get('dense_score', 0.0)) if 'dense_score' in hit else None,
        'dense_rank': int(hit.get('dense_rank', 0)) if 'dense_rank' in hit else None,
        'rrf_score': float(hit.get('rrf_score', 0.0)) if 'rrf_score' in hit else None,
        'rerank_score': float(hit.get('rerank_score', 0.0)) if 'rerank_score' in hit else None,
        'source_hits': list(hit.get('source_hits', []))[:8],
    }

def build_dense_hits_for_variants(dense, variants_by_id):
    flat = []
    for qid, variants in variants_by_id.items():
        for vidx, v in enumerate(variants):
            if not v.get('lexical_only'):
                flat.append((qid, vidx, v))
    dense_hits = defaultdict(dict)
    pbar = tqdm(total=len(flat), desc='Dense decomp search', unit='query', dynamic_ncols=True)
    for start in range(0, len(flat), CFG['dense_query_batch_size']):
        batch = flat[start:start + CFG['dense_query_batch_size']]
        queries = [v['text'] for _, _, v in batch]
        results = dense.search_queries(queries, topk=CFG['dense_topk'])
        for (qid, vidx, _), hits in zip(batch, results):
            dense_hits[int(qid)][int(vidx)] = hits
        pbar.update(len(batch))
    pbar.close()
    return dense_hits

def retrieve_one_query(qid, question, query_plan, variants, lexical, dense, reranker, dense_hits_by_vidx=None, collect_debug=False):
    variant_debug = []
    variant_results = []
    dense_hits_by_vidx = dense_hits_by_vidx or {}
    if dense is not None and not dense_hits_by_vidx:
        dense_queries = [v['text'] for v in variants if not v.get('lexical_only')]
        dense_results = dense.search_queries(dense_queries, topk=CFG['dense_topk']) if dense_queries else []
        dense_iter = iter(dense_results)
    else:
        dense_iter = None

    per_variant_topk = max(CFG['rerank_topk'], CFG['candidate_topk'] // max(len(variants), 1))
    for vidx, variant in enumerate(variants):
        lex_hits = []
        if not variant.get('dense_only'):
            lex_hits = lexical.search(variant['text'], topk=CFG['bm25_topk'])

        d_hits = []
        if not variant.get('lexical_only'):
            if vidx in dense_hits_by_vidx:
                d_hits = dense_hits_by_vidx.get(vidx, [])
            elif dense_iter is not None:
                d_hits = next(dense_iter, [])

        local_rankings = []
        if lex_hits:
            local_rankings.append((lex_hits, variant['weight'], f"lexical:{variant['kind']}"))
        if d_hits:
            local_rankings.append((d_hits, variant['weight'], f"dense:{variant['kind']}"))
        local_fused = weighted_rrf_fuse(local_rankings, k=CFG['rrf_k'], topk=per_variant_topk)
        contexts = fetch_contexts_from_store([h['row_idx'] for h in local_fused], include_text=True)
        score_by_idx = {int(h['row_idx']): h for h in local_fused}
        for c in contexts:
            c.update(score_by_idx.get(int(c['row_idx']), {}))
        reranked = reranker.rerank(variant['text'], contexts, topk=CFG['rerank_topk'])
        variant_results.append({
            'variant': variant,
            'lex_hits': lex_hits,
            'dense_hits': d_hits,
            'fused': local_fused,
            'reranked': reranked,
        })

        if collect_debug:
            variant_debug.append({
                'variant': {k: variant[k] for k in ['kind', 'weight', 'dense_only', 'lexical_only', 'text']},
                'top_lexical': [compact_hit(h) for h in lex_hits[:CFG['diag_topk']]],
                'top_dense': [compact_hit(h) for h in d_hits[:CFG['diag_topk']]],
                'top_fused': [compact_hit(h) for h in local_fused[:CFG['diag_topk']]],
                'top_reranked': [compact_hit(h) for h in reranked[:CFG['diag_topk']]],
            })

    article_candidates = aggregate_article_candidates_from_variants(variant_results, query_plan=query_plan)
    selected_articles, selection_debug = select_article_contexts(article_candidates, query_plan, variants=variants)
    gen_contexts = build_gen_contexts(selected_articles)
    relevant_docs, relevant_articles = make_relevant_lists(selected_articles)

    rec = {
        'id': int(qid),
        'question': question,
        'query_plan': query_plan,
        'atomic_queries': list(query_plan.get('atomic_questions', [])),
        'query_variants': [{k: v[k] for k in ['kind', 'weight', 'dense_only', 'lexical_only', 'text']} for v in variants],
        'candidate_articles_debug': [
            {k: c.get(k, '') for k in ['row_idx', 'chunk_id', 'doc_uid', 'law_id', 'ten_van_ban', 'dieu_so', 'canonical_article_key', 'article_key', 'article_score', 'support_count', 'rerank_score', 'rrf_score', 'original_support', 'atomic_support_count', 'atomic_coverage_count', 'best_variant_kind', 'best_variant_rank', 'best_variant_rerank_score', 'domain_score', 'domain_reasons', 'facet_score', 'best_facet_score', 'facet_evidence_score', 'facet_evidence_scores', 'matched_facets', 'facet_reasons', 'generic_atomic_support', 'law_family_count_before_select', 'source_kinds', 'support_variants', 'source_labels']}
            for c in article_candidates[:CFG['candidate_article_debug_topk']]
        ],
        'selection_debug': selection_debug,
        'article_contexts': [
            {k: c.get(k, '') for k in ['row_idx', 'chunk_id', 'doc_uid', 'law_id', 'ten_van_ban', 'dieu_so', 'canonical_article_key', 'article_key', 'article_score', 'support_count', 'rerank_score', 'rrf_score', 'original_support', 'atomic_support_count', 'atomic_coverage_count', 'best_variant_kind', 'best_variant_rank', 'domain_score', 'domain_reasons', 'facet_score', 'best_facet_score', 'facet_evidence_score', 'facet_evidence_scores', 'matched_facets', 'facet_reasons', 'generic_atomic_support', 'law_family_count_before_select', 'source_kinds', 'selection_reasons']}
            for c in selected_articles
        ],
        'gen_contexts': gen_contexts,
        'relevant_docs': relevant_docs,
        'relevant_articles': relevant_articles,
    }
    if collect_debug:
        rec['variant_debug'] = variant_debug
        all_reranked = []
        for vr in variant_results:
            for h in vr.get('reranked', [])[:CFG['diag_topk']]:
                item = compact_hit(h)
                item['variant_kind'] = vr['variant'].get('kind', '')
                all_reranked.append(item)
        all_reranked.sort(key=lambda x: float(x.get('rerank_score') or -1e9), reverse=True)
        rec['top_reranked_chunks'] = all_reranked[:CFG['diag_topk']]
    return rec

def load_full_test_df():
    with open(READ_PATHS['test_path'], 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data)

def apply_retrieval_shard(df):
    shard_count = CFG['retrieval_shard_count']
    shard_index = CFG['retrieval_shard_index']
    if shard_count <= 1:
        return df.copy()
    out = df[df['id'].astype(int) % shard_count == shard_index].copy()
    print({'retrieval_shard_index': shard_index, 'retrieval_shard_count': shard_count, 'retrieval_shard_size': len(out), 'retrieval_total_before_shard': len(df)})
    return out

def load_retrieval_df():
    return apply_retrieval_shard(load_test_df())

def load_retrieval_components(load_dense=True):
    import faiss
    from FlagEmbedding import BGEM3FlagModel

    lexical = SQLiteLexicalRetriever(READ_PATHS['chunk_store_path'])
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print({'phase': PIPELINE_PHASE, 'device': device, 'cfg': {k: CFG[k] for k in ['lexical_mode', 'bm25_topk', 'dense_topk', 'candidate_topk', 'rerank_topk', 'submit_article_max', 'gen_context_topk', 'retrieval_shard_count', 'retrieval_shard_index']}})
    embed_model = BGEM3FlagModel(EMBED_MODEL, use_fp16=(device == 'cuda')) if load_dense else None
    index = faiss.read_index(READ_PATHS['faiss_path']) if load_dense else None
    if index is not None:
        assert index.ntotal == len(CHUNK_META), (index.ntotal, len(CHUNK_META))
    dense = DenseRetriever(index, CHUNK_META, embed_model) if load_dense else None
    reranker = Reranker()
    return lexical, dense, embed_model, index, reranker

def build_advanced_retrieval_cache():
    test_df = load_retrieval_df()
    plan_cache = load_or_build_query_plan_cache(test_df)
    variants_by_id = {int(row['id']): build_query_variants(str(row['question']), plan_cache[int(row['id'])]) for _, row in test_df.iterrows()}

    lexical, dense, embed_model, index, reranker = load_retrieval_components(load_dense=True)
    dense_hits = build_dense_hits_for_variants(dense, variants_by_id)

    out_path = Path(CFG['retrieval_cache_advanced_path'])
    read_cache_path = find_first(out_path.name)
    if not out_path.exists() and read_cache_path and Path(read_cache_path) != out_path:
        out_path.write_text(Path(read_cache_path).read_text(encoding='utf-8'), encoding='utf-8')

    done_ids = set()
    for rec in read_jsonl(out_path):
        try:
            done_ids.add(int(rec['id']))
        except Exception:
            pass

    remaining = [(int(row['id']), str(row['question'])) for _, row in test_df.iterrows() if int(row['id']) not in done_ids]
    pbar = tqdm(total=len(test_df), initial=len(done_ids & set(test_df['id'].astype(int))), desc='Decomp retrieval cache', unit='query', dynamic_ncols=True)
    article_counts, doc_counts = [], []
    t_start = time.time()
    for qid, question in remaining:
        t0 = time.time()
        rec = retrieve_one_query(
            qid=qid,
            question=question,
            query_plan=plan_cache[qid],
            variants=variants_by_id[qid],
            lexical=lexical,
            dense=None,
            reranker=reranker,
            dense_hits_by_vidx=dense_hits.get(qid, {}),
            collect_debug=False,
        )
        write_jsonl(out_path, rec)
        done_ids.add(qid)
        article_counts.append(len(rec['relevant_articles']))
        doc_counts.append(len(rec['relevant_docs']))
        pbar.update(1)
        if pbar.n % 10 == 0:
            elapsed = max(time.time() - t_start, 1e-9)
            pbar.set_postfix(avg_s=round(elapsed / pbar.n, 3), last_s=round(time.time() - t0, 3), avg_articles=round(float(np.mean(article_counts)), 2) if article_counts else 0)
    pbar.close()

    all_records = read_jsonl(out_path)
    test_ids = set(test_df['id'].astype(int))
    counts = [len(r.get('relevant_articles', [])) for r in all_records if int(r.get('id', -1)) in test_ids]
    multi_doc = [len(r.get('relevant_docs', [])) for r in all_records if int(r.get('id', -1)) in test_ids]
    print('decomp retrieval written:', out_path)
    if counts:
        print({'records': len(counts), 'article_count_mean': float(np.mean(counts)), 'article_count_p50': float(np.percentile(counts, 50)), 'article_count_p90': float(np.percentile(counts, 90)), 'multi_doc_rate': float(np.mean(np.array(multi_doc) >= 2))})

    del lexical, dense, embed_model, index, reranker, dense_hits
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def resolve_smoke_query():
    return resolve_smoke_queries()[0]

def resolve_smoke_queries():
    full_df = load_full_test_df()
    ids = list(SMOKE_QUERY_IDS) if 'SMOKE_QUERY_IDS' in globals() and SMOKE_QUERY_IDS else [int(SMOKE_QUERY_ID)]
    out = []
    for qid in ids:
        smoke_row = full_df[full_df['id'].astype(int) == int(qid)]
        if SMOKE_QUERY_TEXT and len(ids) == 1:
            question = SMOKE_QUERY_TEXT
        elif not smoke_row.empty:
            question = str(smoke_row.iloc[0]['question'])
        else:
            raise ValueError(f'SMOKE_QUERY_ID={qid} not found and SMOKE_QUERY_TEXT is empty')
        out.append((int(qid), normalize_text(question)))
    return out

def article_exact_rank(articles, law_id, dieu_so):
    for rank, article in enumerate(articles, start=1):
        parts = str(article).split('|')
        if len(parts) >= 3 and parts[0] == law_id and parts[-1] == dieu_so:
            return rank
    return None

def build_expected_hit_report(qid, rec):
    final_articles = list(rec.get('relevant_articles', []))
    candidate_articles = [c.get('article_key', '') for c in rec.get('candidate_articles_debug', [])]
    report = {'id': int(qid), 'final_article_count': len(final_articles)}
    if int(qid) in SMOKE_EXPECTED_ARTICLES:
        rows = []
        for law_id, dieu_so in SMOKE_EXPECTED_ARTICLES[int(qid)]:
            rows.append({
                'expected': f'{law_id}|{dieu_so}',
                'final_rank': article_exact_rank(final_articles, law_id, dieu_so),
                'candidate_rank': article_exact_rank(candidate_articles, law_id, dieu_so),
            })
        report['expected_articles'] = rows
        report['final_hits'] = sum(1 for r in rows if r['final_rank'] is not None)
        report['candidate_hits'] = sum(1 for r in rows if r['candidate_rank'] is not None)
    if int(qid) in SMOKE_EXPECTED_DOMAINS:
        domain_rows = []
        haystacks = final_articles
        candidate_haystacks = candidate_articles
        for name, needles in SMOKE_EXPECTED_DOMAINS[int(qid)].items():
            domain_rows.append({
                'domain': name,
                'final_hit': any(any(needle.lower() in str(a).lower() for needle in needles) for a in haystacks),
                'candidate_hit': any(any(needle.lower() in str(a).lower() for needle in needles) for a in candidate_haystacks),
            })
        report['expected_domains'] = domain_rows
    return report

def smoke_debug_path_for_id(qid, multi=False):
    p = Path(CFG['smoke_query_debug_path'])
    if multi or len(resolve_smoke_queries()) > 1:
        return str(p.with_name(f'smoke_query_{int(qid)}_anchor_v2_debug.json'))
    return str(p)

def run_smoke_query():
    smoke_queries = resolve_smoke_queries()
    print({'smoke_query_count': len(smoke_queries), 'smoke_query_ids': [qid for qid, _ in smoke_queries]})

    planner = LLMQueryPlanner()
    planned = []
    for qid, question in smoke_queries:
        print('=' * 100)
        print('SMOKE QUERY')
        print(json.dumps({'id': qid, 'question': question}, ensure_ascii=False, indent=2))
        if qid in SLIDE_REFERENCE_NOTES:
            print('SLIDE_REFERENCE =', json.dumps(SLIDE_REFERENCE_NOTES[qid], ensure_ascii=False, indent=2))
        query_plan = planner.plan(question)
        print('QUERY_PLAN =', json.dumps(query_plan, ensure_ascii=False, indent=2))
        variants = build_query_variants(question, query_plan)
        print('QUERY_VARIANTS =', json.dumps([{k: v[k] for k in ['kind', 'weight', 'dense_only', 'lexical_only', 'text']} for v in variants], ensure_ascii=False, indent=2))
        planned.append((qid, question, query_plan, variants))
    del planner
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    lexical, dense, embed_model, index, reranker = load_retrieval_components(load_dense=True)
    for qid, question, query_plan, variants in planned:
        rec = retrieve_one_query(
            qid=qid,
            question=question,
            query_plan=query_plan,
            variants=variants,
            lexical=lexical,
            dense=dense,
            reranker=reranker,
            dense_hits_by_vidx=None,
            collect_debug=True,
        )
        expected_report = build_expected_hit_report(qid, rec)
        rec['expected_hit_report'] = expected_report
        print('VARIANT_DEBUG =', json.dumps(rec.get('variant_debug', []), ensure_ascii=False, indent=2))
        print('TOP_RERANKED_CHUNKS =', json.dumps(rec.get('top_reranked_chunks', []), ensure_ascii=False, indent=2))
        print('CANDIDATE_ARTICLES_DEBUG =', json.dumps(rec.get('candidate_articles_debug', []), ensure_ascii=False, indent=2))
        print('SELECTION_DEBUG =', json.dumps(rec.get('selection_debug', {}), ensure_ascii=False, indent=2))
        print('FINAL_RELEVANT_DOCS =', json.dumps(rec.get('relevant_docs', []), ensure_ascii=False, indent=2))
        print('FINAL_RELEVANT_ARTICLES =', json.dumps(rec.get('relevant_articles', []), ensure_ascii=False, indent=2))
        print('EXPECTED_HIT_REPORT =', json.dumps(expected_report, ensure_ascii=False, indent=2))
        print({'doc_count': len(rec.get('relevant_docs', [])), 'article_count': len(rec.get('relevant_articles', []))})
        debug_path = smoke_debug_path_for_id(qid, multi=len(planned) > 1)
        Path(debug_path).write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding='utf-8')
        print('smoke debug written:', debug_path)

    del lexical, dense, embed_model, index, reranker
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if PIPELINE_PHASE in {'smoke_query'}:
    run_smoke_query()
elif PIPELINE_PHASE in {'advanced_retrieval', 'all'}:
    build_advanced_retrieval_cache()


In [ ]:
# =========================================================
# 4. GENERATION: IRAC PROMPT + SHARDED PARTIAL RESULTS
# =========================================================
def collect_retrieval_cache_files():
    files = []
    explicit_paths = [Path(CFG['retrieval_cache_advanced_path']), Path(READ_PATHS['retrieval_cache_advanced_path'])]
    for p in explicit_paths:
        if p.exists() and p.is_file():
            files.append(p)
    files.extend(find_all('retrieval_cache_decomp_anchor_v2_shard*_of_*__*.jsonl'))
    files.extend(find_all('retrieval_cache_decomp_anchor_v2__*.jsonl'))
    out_root = OUT_DIR.resolve().as_posix()
    files = sorted(set(files), key=lambda p: (p.resolve().as_posix().startswith(out_root), p.as_posix()))
    return files

def load_advanced_retrieval_records():
    files = collect_retrieval_cache_files()
    assert files, f"Missing retrieval cache files. Expected {CFG['retrieval_cache_advanced_path']} or attached retrieval_cache_decomp_anchor_v2*.jsonl"
    print('retrieval cache files:', [str(f) for f in files])
    by_id = {}
    for f in files:
        for rec in read_jsonl(f):
            try:
                qid = int(rec['id'])
            except Exception:
                continue
            by_id[qid] = rec
    return [by_id[qid] for qid in sorted(by_id)]

def strip_leading_prompt_echo(text):
    text = normalize_text(text)
    patterns = [
        r'(?is)^system\s*.*?user\s*',
        r'(?is)^assistant\s*',
        r'(?is)^trả lời\s*:?',
    ]
    for pat in patterns:
        text = re.sub(pat, '', text).strip()
    return re.sub(r'^\s*[-–—]\s*', '', text).strip()

def allowed_article_numbers(relevant_articles):
    allowed = set()
    for a in relevant_articles:
        for m in re.finditer(r'Điều\s+\d+[A-Za-zÀ-ỹ]*', str(a), flags=re.IGNORECASE):
            allowed.add(normalize_text(m.group(0)).lower())
    return allowed

def postprocess_answer_citations(answer, relevant_articles):
    answer = strip_leading_prompt_echo(answer)
    allowed = allowed_article_numbers(relevant_articles)
    mentioned = {normalize_text(m.group(0)).lower() for m in re.finditer(r'Điều\s+\d+[A-Za-zÀ-ỹ]*', answer, flags=re.IGNORECASE)}
    missing = sorted(mentioned - allowed)
    if missing:
        answer += '\n\nLưu ý: Các nhận định trên chỉ dựa trên các căn cứ đã truy xuất; chưa đủ căn cứ để khẳng định các điều luật ngoài danh sách liên quan.'
    if allowed and not re.search(r'Điều\s+\d+', answer, flags=re.IGNORECASE):
        top_cites = []
        for a in relevant_articles[:5]:
            parts = str(a).split('|')
            if len(parts) >= 3:
                top_cites.append(parts[-1])
        if top_cites:
            answer += '\n\nCăn cứ tham khảo: ' + ', '.join(dict.fromkeys(top_cites)) + '.'
    return answer

class QwenIRACGenerator:
    def __init__(self, model_name=GEN_MODEL):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        load_kwargs = {'trust_remote_code': True}
        if device == 'cuda' and CFG.get('gen_load_in_4bit', True):
            from transformers import BitsAndBytesConfig
            load_kwargs.update({
                'device_map': 'auto',
                'torch_dtype': torch.float16,
                'quantization_config': BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_type='nf4',
                    bnb_4bit_use_double_quant=True,
                ),
            })
        elif device == 'cuda':
            load_kwargs.update({'torch_dtype': torch.float16, 'device_map': 'auto'})
        else:
            load_kwargs.update({'torch_dtype': torch.float32})
        print({'generator_device': device, 'load_in_4bit': bool(device == 'cuda' and CFG.get('gen_load_in_4bit', True)), 'gen_context_topk': CFG['gen_context_topk'], 'gen_chunk_char_limit': CFG['gen_chunk_char_limit'], 'gen_max_input_tokens': CFG['gen_max_input_tokens']})
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        self.model.eval()

    def build_prompt(self, question, contexts, relevant_articles):
        ctx_blocks = []
        for i, c in enumerate(contexts, start=1):
            meta = ' | '.join(str(c.get(k, '')) for k in ['law_id', 'ten_van_ban', 'dieu_so'])
            text = str(c.get('chunk_text', ''))[:CFG['gen_chunk_char_limit']]
            ctx_blocks.append(f'[{i}] {meta}\n{text}')
        allowed = '\n'.join('- ' + str(a) for a in relevant_articles[:CFG['article_context_topk']])
        system = (
            'Bạn là trợ lý pháp lý AI cho doanh nghiệp SME tại Việt Nam. '
            'Chỉ trả lời dựa trên ngữ cảnh và danh sách căn cứ được cung cấp. '
            'Không bịa văn bản, điều luật, khoản hoặc nguồn tham chiếu. '
            'Nếu thiếu căn cứ, nói rõ là chưa đủ căn cứ.'
        )
        user = (
            f'Câu hỏi: {question}\n\n'
            'Danh sách điều luật được phép viện dẫn:\n'
            f'{allowed}\n\n'
            'Ngữ cảnh pháp lý:\n'
            f"{chr(10).join(ctx_blocks)}\n\n"
            'Yêu cầu trả lời:\n'
            '- Trả lời bằng tiếng Việt, ngắn gọn nhưng đủ ý.\n'
            '- Dùng cấu trúc IRAC ngắn: Vấn đề, Quy định, Áp dụng, Kết luận.\n'
            '- Luôn nêu Điều X khi có căn cứ trong danh sách được phép viện dẫn.\n'
            '- Không nhắc lại toàn bộ ngữ cảnh, không trích dẫn điều ngoài danh sách.'
        )
        if hasattr(self.tokenizer, 'apply_chat_template'):
            return self.tokenizer.apply_chat_template([
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': user},
            ], tokenize=False, add_generation_prompt=True)
        return system + '\n\n' + user + '\n\nTrả lời:'

    def generate(self, question, contexts, relevant_articles):
        prompt = self.build_prompt(question, contexts, relevant_articles)
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=CFG['gen_max_input_tokens']).to(self.model.device)
        temperature = float(os.environ.get('GEN_TEMPERATURE', '0.1'))
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=int(os.environ.get('GEN_MAX_NEW_TOKENS', '448')),
                temperature=temperature,
                top_p=float(os.environ.get('GEN_TOP_P', '0.9')),
                do_sample=temperature > 0,
                repetition_penalty=1.02,
                use_cache=os.environ.get('GEN_USE_CACHE', '1') == '1',
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        gen_ids = out[0][inputs['input_ids'].shape[1]:]
        raw_text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        return strip_leading_prompt_echo(raw_text)

def shard_partial_path():
    count = CFG['gen_shard_count']
    index = CFG['gen_shard_index']
    answer_mode = 'answer' if bool(CFG.get('answer', True)) else 'noanswer'
    if count <= 1:
        return OUT_DIR / f"results_partial_decomp_anchor_v2_{answer_mode}__{EMBED_MODEL.replace('/','_')}__{GEN_MODEL.replace('/','_')}.jsonl"
    return OUT_DIR / f'results_partial_decomp_anchor_v2_{answer_mode}_shard{index}_of_{count}.jsonl'

def load_progress():
    p = Path(READ_PATHS['progress_path'])
    if p.exists():
        return json.loads(p.read_text(encoding='utf-8'))
    return {'done_ids': [], 'mode': RUN_MODE, 'shard_count': CFG['gen_shard_count'], 'shard_index': CFG['gen_shard_index']}

def save_progress(state):
    Path(CFG['progress_path']).write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')

def build_generation_shard():
    records = load_advanced_retrieval_records()
    by_id = {int(r['id']): r for r in records}
    test_df = load_test_df()
    missing = [int(i) for i in test_df['id'].tolist() if int(i) not in by_id]
    assert not missing, f'Missing advanced retrieval records for ids: {missing[:10]}'

    shard_count = CFG['gen_shard_count']
    shard_index = CFG['gen_shard_index']
    shard_ids = {int(i) for i in test_df['id'].tolist() if int(i) % shard_count == shard_index}
    print({'phase': 'generate', 'shard_index': shard_index, 'shard_count': shard_count, 'shard_size': len(shard_ids), 'answer': bool(CFG.get('answer', True))})

    should_generate_answer = bool(CFG.get('answer', True))
    generator = None
    if should_generate_answer:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        generator = QwenIRACGenerator()
    else:
        print('ANSWER=0: skipping LLM answer generation; every answer field will be empty.')

    partial_path = shard_partial_path()
    read_partial_path = find_first(partial_path.name)
    if not partial_path.exists() and read_partial_path and Path(read_partial_path) != partial_path:
        partial_path.write_text(Path(read_partial_path).read_text(encoding='utf-8'), encoding='utf-8')
    done_ids = set()
    if partial_path.exists():
        for rec in read_jsonl(partial_path):
            try:
                done_ids.add(int(rec['id']))
            except Exception:
                pass
    progress = load_progress()

    pbar = tqdm(total=len(shard_ids), initial=len(done_ids & shard_ids), desc='Generate shard answers', unit='query', dynamic_ncols=True)
    for _, row in test_df.iterrows():
        qid = int(row['id'])
        if qid not in shard_ids or qid in done_ids:
            continue
        cached = by_id[qid]
        relevant_articles = cached.get('relevant_articles', [])
        if should_generate_answer:
            try:
                answer = generator.generate(str(row['question']), cached.get('gen_contexts', []), relevant_articles)
            except torch.cuda.OutOfMemoryError:
                print(f'OOM on id={qid}; retrying with shorter contexts')
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                short_contexts = []
                for c in cached.get('gen_contexts', [])[:2]:
                    c2 = dict(c)
                    c2['chunk_text'] = str(c2.get('chunk_text', ''))[:600]
                    short_contexts.append(c2)
                answer = generator.generate(str(row['question']), short_contexts, relevant_articles[:8])
            answer = postprocess_answer_citations(answer, relevant_articles)
        else:
            answer = ''
        rec = {
            'id': qid,
            'question': str(row['question']),
            'answer': answer,
            'relevant_docs': cached.get('relevant_docs', []),
            'relevant_articles': relevant_articles,
        }
        write_jsonl(partial_path, rec)
        done_ids.add(qid)
        progress['done_ids'] = sorted(done_ids)
        progress['shard_count'] = shard_count
        progress['shard_index'] = shard_index
        save_progress(progress)
        pbar.update(1)
        if len(done_ids) % SAVE_EVERY == 0:
            print(f'checkpoint saved, shard_done={len(done_ids)}')
    pbar.close()
    print('partial shard written:', partial_path)

if PIPELINE_PHASE in {'generate', 'all'}:
    build_generation_shard()


In [ ]:
# =========================================================
# 5. MERGE SHARDS + VALIDATION + ZIP
# =========================================================
def collect_partial_result_files():
    files = []
    files.extend(find_all('results_partial_decomp_anchor_v2_*_shard*_of_*.jsonl'))
    files.extend(find_all('results_partial_decomp_anchor_v2_*__*.jsonl'))
    # Process attached/input files first, then current output files so local reruns override older shards.
    out_root = OUT_DIR.resolve().as_posix()
    files = sorted(set(files), key=lambda p: (p.resolve().as_posix().startswith(out_root), p.as_posix()))
    return files

def validate_submission(path=None):
    path = Path(path or CFG['results_path'])
    assert path.exists(), f'Missing results file: {path}'
    data = json.loads(path.read_text(encoding='utf-8'))
    assert isinstance(data, list)
    test_df = load_test_df()
    expected_ids = set(int(x) for x in test_df['id'].tolist())
    got_ids = set(int(r.get('id')) for r in data)
    assert expected_ids == got_ids, {'missing': sorted(expected_ids - got_ids)[:20], 'extra': sorted(got_ids - expected_ids)[:20]}
    required = {'id', 'question', 'answer', 'relevant_docs', 'relevant_articles'}
    bad = [r.get('id') for r in data if set(r.keys()) != required]
    assert not bad, f'Records with wrong schema: {bad[:10]}'
    for r in data[:5]:
        assert isinstance(r['relevant_docs'], list)
        assert isinstance(r['relevant_articles'], list)
    print({'records': len(data), 'schema_ok': True, 'ids_ok': True})

def merge_generation_shards():
    test_df = load_test_df()
    expected_ids = set(int(x) for x in test_df['id'].tolist())
    files = collect_partial_result_files()
    assert files, 'No partial result files found. Attach shard outputs or run PIPELINE_PHASE=generate first.'
    print('partial files:', [str(f) for f in files])
    by_id = {}
    for f in files:
        for rec in read_jsonl(f):
            try:
                qid = int(rec['id'])
            except Exception:
                continue
            if qid in expected_ids:
                by_id[qid] = {
                    'id': qid,
                    'question': str(rec.get('question', '')),
                    'answer': str(rec.get('answer', '')),
                    'relevant_docs': list(rec.get('relevant_docs', [])),
                    'relevant_articles': list(rec.get('relevant_articles', [])),
                }
    missing = sorted(expected_ids - set(by_id))
    assert not missing, {'missing_ids': missing[:50], 'missing_count': len(missing)}
    final = [by_id[qid] for qid in sorted(expected_ids)]
    Path(CFG['results_path']).write_text(json.dumps(final, ensure_ascii=False, indent=2), encoding='utf-8')
    with zipfile.ZipFile(CFG['zip_path'], 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(CFG['results_path'], arcname='results.json')
    validate_submission(CFG['results_path'])
    print('results written:', CFG['results_path'])
    print('zip written:', CFG['zip_path'])

if PIPELINE_PHASE in {'merge', 'all'}:
    merge_generation_shards()
elif Path(CFG['results_path']).exists():
    validate_submission(CFG['results_path'])
